# AeroVerify-VLM v2: Evidence-Grounded, Reliability-Weighted VQA for UAV Imagery

**Version 2.** Redesigned after run 1 on RSVQA-LR. Each change addresses a measured failure:

| Run-1 finding | v2 design |
|---|---|
| 10 m satellite tiles: both VLMs ≈ most-common answer; crops add no information | **FloodNet-VQA (real UAV, 4000×3000 px)** is the primary benchmark. RSVQA-LR is kept as the *low-resolution limit* case |
| Upscaled crops hurt SmolVLM | **Resolution-gated native tiles**: zoom only when tiles carry more pixels than the model's input |
| Detector evidence helped Moondream2 most; SmolVLM had none | **Shared grounding module** for both VLMs, plus **grounded instance-level counting** |
| Moondream2 says "yes" to 92–97 % of presence questions; correlated sources cannot outvote a shared bias | **Contrastive / decomposed probes** that avoid yes/no tokens |
| Plurality voting gives 3–4 confidence levels | **Reliability-weighted fusion** (weights from DEV accuracy), giving a posterior confidence and smooth risk–coverage |
| E1 penalised by unparseable free text | E1 uses the standard short-answer VQA suffix. **E2 is the headline baseline** |
| No cost control | **AeroVerify-Lite**: an early-exit cascade |

**Experiments:** E1 direct · E2 structured · E3 region · **E4 AeroVerify** (and Lite) · E5 ablations (computed from cached evidence) · E6 efficiency · E7 UAV-specific robustness (altitude, motion blur, haze, JPEG).

**Outputs:** `outputs_v2/figures/*.pdf` (IEEE widths, ≥7 pt text, embedded fonts) and `outputs_v2/tables/*.tex` (booktabs, paste into LaTeX).

> Run headless with `code/run_experiments.sh` (see `README.md`). Settings can be overridden with `AV_*` environment variables.

## 1 · Parameters
Edit here or override with `AV_*` environment variables. `AV_DRY_RUN=1` runs everything on synthetic data with mock models.

In [ ]:
# ── Parameters (override any with environment variables AV_*) ─────────────────
import os, json
from pathlib import Path

def _env(name, default, cast=str):
    v = os.environ.get(name)
    if v is None or v == "":
        return default
    if cast is bool:
        return v.strip().lower() in ("1", "true", "yes", "y")
    if cast is list:
        return [s.strip() for s in v.split(",") if s.strip()]
    return cast(v)

DRY_RUN       = _env("AV_DRY_RUN", False, bool)          # synthetic data + mock VLMs (smoke test)
PROJECT_ROOT  = _env("PROJECT_ROOT", str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))   # repo root
STAGES        = _env("AV_STAGES", ["data", "run", "analyze"], list)
DATASETS      = _env("AV_DATASETS", ["floodnet", "rsvqa_lr"], list)   # primary first
MODELS_TO_RUN = _env("AV_MODELS", ["moondream2", "smolvlm"], list)
MODELS_TO_ANALYZE = _env("AV_ANALYZE_MODELS", ["moondream2", "smolvlm"], list)
RUN_ROBUSTNESS = _env("AV_ROBUSTNESS", True, bool)
ROBUST_DATASETS = ["floodnet"]                               # UAV perturbations are applied to UAV imagery

# Samples (DEV → reliability weights + thresholds only; EVAL → every reported number)
SAMPLE_SIZES = {
    "floodnet": dict(dev=_env("AV_FN_DEV", 60, int), eval=_env("AV_FN_EVAL", 300, int)),
    "rsvqa_lr": dict(dev=40, eval=160),                    # identical to run 1
}
ROBUST_N_IMAGES = _env("AV_N_ROBUST", 40, int)
SEED = 2026

# Evidence
MODEL_SIDE      = 768     # nominal VLM input side (px) used by the resolution gate
GATE_MIN_ZOOM   = 1.0     # native tiles are used only if tile_side / MODEL_SIDE ≥ this
GRID            = 2       # 2×2 tiles
LR_CROP_SIZE    = 512     # E3 baseline on RSVQA-LR upsamples 128-px crops (gate keeps them out of E4)
MAX_BOXES       = 16      # grounded counting: instances verified one by one
DET_MAX_OBJECTS = 100

# Decision rule
MIN_COVERAGE = 0.80       # τ = most accurate threshold on DEV that still answers ≥ 80 %

# E7 robustness (UAV-specific)
PERTURBATIONS = {
    "altitude":    [0.5, 0.25, 0.125],   # down-sample factor (simulates higher flight altitude)
    "motion_blur": [15, 31, 61],         # horizontal blur length at native resolution (px)
    "haze":        [0.3, 0.5, 0.7],      # blend with atmospheric light
    "jpeg":        [30, 15, 5],          # JPEG quality (video link compression)
}

MODEL_SPECS = {
    "moondream2": dict(hf_id="vikhyatk/moondream2", revision="2025-06-21", label="Moondream2"),
    "smolvlm":    dict(hf_id="HuggingFaceTB/SmolVLM-Instruct", revision="main", label="SmolVLM",
                       longest_edge=768),
}
GROUNDER = dict(hf_id="vikhyatk/moondream2", revision="2025-06-21")   # shared grounding module
MAX_NEW_TOKENS = 24
FLOODNET_HF = "takara-ai/FloodNet_2021-Track_2_Dataset_HF"

print(json.dumps(dict(DRY_RUN=DRY_RUN, STAGES=STAGES, DATASETS=DATASETS, MODELS_TO_RUN=MODELS_TO_RUN,
                      SAMPLE_SIZES=SAMPLE_SIZES, RUN_ROBUSTNESS=RUN_ROBUSTNESS), indent=1))

## 2 · Environment

In [ ]:
# ── Environment ───────────────────────────────────────────────────────────────
import sys, re, time, math, random, hashlib, zipfile, gc, ast, urllib.request
from pathlib import Path
from dataclasses import dataclass
from collections import Counter, defaultdict, OrderedDict

ROOT = Path(PROJECT_ROOT) / ("dry_run_v2" if DRY_RUN else "")
OUT_DIR   = ROOT / "outputs_v2"
FIG_DIR, TAB_DIR = OUT_DIR / "figures", OUT_DIR / "tables"
CACHE_DIR, PRED_DIR = OUT_DIR / "cache", OUT_DIR / "predictions"
for d in (FIG_DIR, TAB_DIR, CACHE_DIR, PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)
def data_dir(ds):
    p = ROOT / "data" / (f"synthetic_{ds}" if DRY_RUN else ds); p.mkdir(parents=True, exist_ok=True); return p

os.environ.setdefault("HF_HOME", str(Path(PROJECT_ROOT) / ".cache" / "huggingface"))
os.environ.setdefault("TORCH_HOME", str(Path(PROJECT_ROOT) / ".cache" / "torch"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
from PIL import Image, ImageFilter, ImageEnhance, ImageDraw
Image.MAX_IMAGE_PIXELS = None
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k): return x
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
try:
    import torch
    torch.manual_seed(SEED); HAS_CUDA = torch.cuda.is_available()
except ImportError:
    torch, HAS_CUDA = None, False

# %%stage <name>: run a cell only when that stage is selected
try:
    from IPython.core.magic import register_cell_magic
    @register_cell_magic
    def stage(line, cell):
        if line.strip() in STAGES:
            get_ipython().run_cell(cell).raise_error()
        else:
            print(f"(stage '{line.strip()}' not selected — cell skipped)")
except (ImportError, NameError):
    pass

print("Python :", sys.version.split()[0], "|", sys.executable)
print("Outputs:", OUT_DIR)
if torch is not None:
    print("PyTorch:", torch.__version__, "| CUDA:", HAS_CUDA, "|", torch.cuda.get_device_name(0) if HAS_CUDA else "-")
if not DRY_RUN and "run" in STAGES and not HAS_CUDA:
    raise RuntimeError("No CUDA GPU visible — select the project .venv kernel or set AV_DRY_RUN=1")

In [ ]:
# ── Figure style: IEEE journal (text ≥ 7 pt at print size) ────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRIDC, AXISC, SURFACE, BAND = "#e1e0d9", "#c3c2b7", "#ffffff", "#f6f5f1"
METHOD_COLORS  = {"E1": "#eb6834", "E2": "#1baf7a", "E3": "#eda100", "E4": "#2a78d6", "Lite": "#6250d6"}
METHOD_MARKERS = {"E1": "o", "E2": "s", "E3": "^", "E4": "D", "Lite": "P"}
METHOD_LABELS  = {"E1": "E1 Direct", "E2": "E2 Structured", "E3": "E3 Region", "E4": "E4 AeroVerify",
                  "Lite": "AeroVerify-Lite", "majority": "Most-common answer", "blind": "Question-only"}
BASELINE_COLOR = MUTED
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
POS, NEG, MID = "#2a78d6", "#e34948", "#f0efec"
DS_LABEL = {"floodnet": "FloodNet-VQA (UAV)", "rsvqa_lr": "RSVQA-LR (satellite, 10 m)"}
DS_SHORT = {"floodnet": "FloodNet", "rsvqa_lr": "RSVQA-LR"}

IEEE_1COL, IEEE_2COL = 3.5, 7.16
FS = 8                                            # base font size (pt) at print size
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "font.family": "DejaVu Sans", "font.size": FS,
    "axes.titlesize": FS + 0.5, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 4,
    "axes.labelsize": FS, "axes.labelcolor": INK2,
    "xtick.labelsize": FS - 0.5, "ytick.labelsize": FS - 0.5, "xtick.color": INK2, "ytick.color": INK2,
    "axes.edgecolor": AXISC, "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.5, "axes.axisbelow": True,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "legend.fontsize": FS - 0.5, "legend.frameon": False, "legend.handletextpad": 0.4,
    "legend.columnspacing": 1.2, "legend.handlelength": 1.4,
    "lines.linewidth": 1.4, "lines.markersize": 5,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
PCT = mticker.FuncFormatter(lambda v, _: f"{v:g}%")

def savefig(fig, name, note=None):
    """Vector PDF (paper) + 600-dpi PNG (slides), then show inline."""
    if DRY_RUN:
        bb = fig.get_tightbbox(fig.canvas.get_renderer()); W, H = fig.get_size_inches()
        fig.text((bb.x0 + bb.width / 2) / W, (bb.y0 - 0.04) / H,
                 "DRY RUN · synthetic data + mock models — numbers are NOT results",
                 ha="center", va="top", fontsize=7, color=NEG)
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{name}.{ext}")
    plt.show()
    print(f"saved → {FIG_DIR / name}.pdf" + (f"   ({note})" if note else ""))

def panel_label(ax, text, x=-0.02, y=1.02):
    ax.text(x, y, text, transform=ax.transAxes, ha="right", va="bottom", fontsize=FS + 1, fontweight="bold")

## 3 · Question routing and answer normalisation
Rule-based routers (no labels used) and parsers that map free text to each dataset's fixed answers. Self-tests run at the end of the cell.

In [ ]:
# ── Question understanding & answer normalisation (both datasets) ─────────────
# Every free-text answer is mapped to the dataset's fixed labels; unmappable → invalid (scored wrong).
BIN_KEYS = ["0", "1-10", "11-100", "101-1000", ">1000"]
COUNT_BINS = ["0", "1–10", "11–100", "101–1000", ">1000"]
FLOODED, NONFLOODED, MIXED = "flooded", "non flooded", "flooded,non flooded"

# type → answer kind  (yesno · cat · count_bin · count)
TYPES = {
    "rsvqa_lr": {"presence": "yesno", "comp": "yesno", "count": "count_bin", "rural_urban": "cat"},
    "floodnet": {"yesno": "yesno", "condition": "cat", "count_simple": "count", "count_complex": "count"},
}
TYPE_LABEL = {"presence": "Presence", "comp": "Comparison", "count": "Count", "rural_urban": "Rural/Urban",
              "yesno": "Yes/No", "condition": "Condition", "count_simple": "Simple count",
              "count_complex": "Complex count", "OA": "Overall (OA)", "AA": "Average (AA)"}
FLOODNET_TYPEMAP = {"Simple_Counting": "count_simple", "Complex_Counting": "count_complex",
                    "Condition_Recognition": "condition", "Yes_No": "yesno"}

REL_RE   = re.compile(r"\b(?:at the |on the |to the )?(top of|bottom of|left of|right of|next to)\b")
SMALL_RE = re.compile(r"\bsmall\b")
SIZE_RE  = re.compile(r"\b(small|medium|large)\b")
_TAIL    = r"(?:\s+(?:in|on|of) (?:the|this|given) (?:image|picture|area|region|scene))?\s*\??\s*$"

def _norm(q): return re.sub(r"\s+", " ", str(q).strip().lower())
def _strip_article(s): return re.sub(r"^(?:a|an|the|some|any)\s+", "", s.strip())

# ---------- RSVQA-LR (as run 1, plus "is a X present") ----------
def route_lr(q):
    s = _norm(q)
    if "rural" in s or "urban" in s: return "rural_urban"
    if re.match(r"^(how many|what is the number of|what is the amount of|what is the count of)\b", s): return "count"
    if re.search(r"\b(more|less|fewer|greater|smaller|larger|bigger|higher|lower)\b.*\bthan\b", s) \
       or re.search(r"\bequal\b|\bas many\b|\bsame number\b", s): return "comp"
    return "presence"

def presence_target(q):
    s = _norm(q)
    m = re.match(r"^(?:is|are) there (.+?)" + _TAIL, s) or \
        re.match(r"^(?:is|are) (.+?) (?:present|visible)" + _TAIL, s) or \
        re.match(r"^(?:can you see|do you see|does the image (?:contain|show)) (.+?)" + _TAIL, s)
    return _strip_article(m.group(1)) if m else None

def count_target(q):
    s = _norm(q)
    m = re.match(r"^(?:how many|what is the number of|what is the amount of|what is the count of) (.+?)"
                 r"(?:\s+(?:are|is) there|\s+(?:are|is) present|\s+(?:are|is)|\s+(?:in|on) (?:the|this|given) "
                 r"(?:image|picture|scene))*\s*\??\s*$", s)
    return _strip_article(m.group(1)) if m else None

def comp_parts(q):
    s = _norm(q)
    m = re.match(r"^(?:is|are) there (more|less|fewer) (.+?) than (.+?)" + _TAIL, s)
    if m:
        return dict(op="more" if m.group(1) == "more" else "less", a=_strip_article(m.group(2)), b=_strip_article(m.group(3)))
    m = re.match(r"^is the number of (.+?) (equal to|the same as|greater than|larger than|bigger than|higher than|"
                 r"more than|less than|smaller than|lower than|fewer than) the number of (.+?)" + _TAIL, s)
    if m:
        rel = m.group(2)
        op = "equal" if rel in ("equal to", "the same as") else \
             "more" if rel in ("greater than", "larger than", "bigger than", "higher than", "more than") else "less"
        return dict(op=op, a=_strip_article(m.group(1)), b=_strip_article(m.group(3)))
    m = re.match(r"^are there as many (.+?) as (.+?)" + _TAIL, s)
    return dict(op="equal", a=_strip_article(m.group(1)), b=_strip_article(m.group(2))) if m else None

def relational_subject(target):
    if not target: return None
    m = REL_RE.search(target)
    return _strip_article(target[:m.start()].strip()) if m else None

# ---------- FloodNet-VQA ----------
FN_NOUNS = ["building", "road", "vehicle", "car", "tree", "pool", "house"]
def route_fn(q):
    s = _norm(q)
    if s.startswith("how many") or "number of" in s:
        return "count_complex" if re.search(r"\b(non[ -]?flooded|not flooded|flooded)\b", s) else "count_simple"
    if "condition" in s: return "condition"
    return "yesno"

def fn_attr(q):
    s = _norm(q)
    if re.search(r"\bnon[ -]?flooded\b|\bnot flooded\b", s): return NONFLOODED
    if re.search(r"\bflooded\b", s): return FLOODED
    return None

def fn_object(q):
    s = _norm(q)
    for n in FN_NOUNS:
        if re.search(rf"\b{n}s?\b", s):
            return {"car": "vehicle", "house": "building"}.get(n, n)
    return "building"

def fn_yesno_parts(q):
    """'Is the entire road (non) flooded?' → {'obj','polarity','quant'}; None if another form."""
    s = _norm(q)
    m = re.match(r"^(?:is|are) (?:the )?(entire|whole|all(?: of)?(?: the)?|any|some(?: of)?(?: the)?)?\s*(\w+?)s? "
                 r"(?:area |areas |surface )?(non[ -]?flooded|not flooded|flooded)\s*\??$", s)
    if not m:
        m2 = re.match(r"^(?:is|are) there (?:any |a )?(non[ -]?flooded|flooded) (\w+?)s?\s*\??$", s)
        if m2:
            return dict(obj=m2.group(2), polarity=NONFLOODED if "non" in m2.group(1) else FLOODED, quant="any")
        return None
    quant = "any" if (m.group(1) or "").startswith(("any", "some")) else "entire"
    pol = NONFLOODED if re.match(r"non|not", m.group(3)) else FLOODED
    return dict(obj=m.group(2), polarity=pol, quant=quant)

def cond_subtype(q):
    return "road" if "road" in _norm(q) else "image"

# ---------- generic ----------
def route(q, ds): return route_lr(q) if ds == "rsvqa_lr" else route_fn(q)
def kind_of(ds, t): return TYPES[ds][t]

def question_tags(q, ds):
    s = _norm(q)
    if ds == "rsvqa_lr":
        sg = "spatial" if REL_RE.search(s) else ("small-object" if SMALL_RE.search(s) else "other")
    else:
        sg = "attribute" if fn_attr(q) else "plain"
    return dict(small=bool(SMALL_RE.search(s)), relational=bool(REL_RE.search(s)), subgroup=sg)

_NUM_WORDS = {"zero": 0, "no": 0, "none": 0, "one": 1, "a single": 1, "single": 1, "two": 2, "a couple of": 2,
              "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
              "eleven": 11, "twelve": 12, "a dozen": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15,
              "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19, "twenty": 20,
              "thirty": 30, "forty": 40, "fifty": 50, "hundred": 100, "a hundred": 100, "thousand": 1000}
_NUM_WORD_RE = re.compile(r"\b(" + "|".join(sorted(map(re.escape, _NUM_WORDS), key=len, reverse=True)) + r")\b")

def count_bin(n):
    if n is None: return None
    n = int(n)
    return "0" if n <= 0 else "1-10" if n <= 10 else "11-100" if n <= 100 else "101-1000" if n <= 1000 else ">1000"

def parse_count(text):
    s = str(text).lower()
    m = re.search(r"\d[\d,]*(?:\.\d+)?", s)
    if m:
        try: return int(round(float(m.group(0).replace(",", ""))))
        except ValueError: pass
    m = _NUM_WORD_RE.search(s)
    if m: return _NUM_WORDS[m.group(1)]
    if re.search(r"\b(not any|nothing|there are no|there is no)\b", s): return 0
    return None

def parse_yesno(text):
    s = str(text).lower()
    m = re.search(r"\b(yes|no|true|false|yeah|yep|nope|correct|incorrect|present|absent)\b", s)
    if m:
        return "yes" if m.group(1) in ("yes", "true", "yeah", "yep", "correct", "present") else "no"
    if re.search(r"\b(there (?:is|are) no|not|none|cannot see|can't see)\b", s): return "no"
    if re.search(r"\bthere (?:is|are)\b|\bi can see\b", s): return "yes"
    return None

_RU = {"rural": "rural", "countryside": "rural", "agricultural": "rural", "farmland": "rural", "village": "rural",
       "urban": "urban", "city": "urban", "town": "urban", "suburban": "urban", "residential": "urban"}
def parse_rural_urban(text):
    s = str(text).lower()
    hits = [(m.start(), _RU[m.group(1)]) for m in re.finditer(r"\b(" + "|".join(_RU) + r")\b", s)]
    return min(hits)[1] if hits else None

def parse_condition(text):
    """→ flooded | non flooded | mixed | None"""
    s = str(text).lower().replace("-", " ")
    if re.search(r"\b(partial|partially|partly|some of|mixed|both|half)\b", s): return "mixed"
    if re.search(r"\b(non flooded|not flooded|unflooded|dry|no flood\w*|none|good|normal|fine|clear|intact|passable|undamaged)\b", s):
        return NONFLOODED
    if re.search(r"\b(flooded|flood|under water|underwater|inundated|submerged|waterlogged|impassable|all of it|all)\b", s):
        return FLOODED
    return None

def parse_instance(text):
    """instance check 'flooded or dry' → flooded / non flooded (falls back to yes/no)."""
    c = parse_condition(text)
    if c in (FLOODED, NONFLOODED): return c
    yn = parse_yesno(text)
    return None if yn is None else (FLOODED if yn == "yes" else NONFLOODED)

def parse_amount(text):
    """'none / some / all' probe → condition label"""
    s = str(text).lower()
    m = re.search(r"\b(none|nothing|no part|some|part|partly|partially|half|most|all|entire|entirely|whole)\b", s)
    if not m: return parse_condition(text)
    w = m.group(1)
    return NONFLOODED if w in ("none", "nothing", "no part") else FLOODED if w in ("all", "entire", "entirely", "whole") else "mixed"

# answer space per condition sub-type; re-estimated from the training pool in the sampling cell
COND_LABELS = {"road": [MIXED, FLOODED, NONFLOODED], "image": [FLOODED, NONFLOODED]}

def cond_to_label(c, subtype):
    if c is None: return None
    if c == "mixed": return MIXED if MIXED in COND_LABELS[subtype] else FLOODED
    return c

def normalize_answer(text, ds, t, q=None):
    if text is None: return None
    k = kind_of(ds, t)
    if k == "yesno": return parse_yesno(text)
    if k == "count_bin": return count_bin(parse_count(text))
    if k == "count":
        n = parse_count(text); return None if n is None else str(max(0, n))
    if t == "rural_urban": return parse_rural_urban(text)
    if t == "condition": return cond_to_label(parse_condition(text), cond_subtype(q or ""))
    raise ValueError((ds, t))

_GT_BINS = {"0": "0", "between 1 and 10": "1-10", "between 11 and 100": "11-100",
            "between 101 and 1000": "101-1000", "more than 1000": ">1000"}
def gt_label(answer, ds, t):
    a = str(answer).strip().lower()
    if ds == "rsvqa_lr":
        return (_GT_BINS.get(a) or count_bin(parse_count(a))) if t == "count" else a
    if t in ("count_simple", "count_complex"):
        n = parse_count(a); return None if n is None else str(n)
    if t == "condition":
        a = re.sub(r"\s+", " ", a.replace("-", " ").replace("_", " "))
        return re.sub(r"\s*,\s*", ",", a)
    return a

# ── self-tests ───────────────────────────────────────────────────────────────
assert route_lr("Is it a rural or an urban area?") == "rural_urban" and route_lr("How many roads are there?") == "count"
assert route_lr("Is the number of roads equal to the number of water areas?") == "comp"
assert presence_target("Is a residential building present?") == "residential building"
assert count_target("How many buildings are in the image?") == "buildings"
assert count_target("How many large buildings are there?") == "large buildings"
assert comp_parts("Are there less roads than buildings?") == dict(op="less", a="roads", b="buildings")
assert route_fn("How many buildings are in the image?") == "count_simple"
assert route_fn("How many buildings are non flooded in this image?") == "count_complex"
assert route_fn("What is the condition of the road in this image?") == "condition"
assert route_fn("Is the entire road non flooded?") == "yesno"
assert fn_attr("How many buildings are non flooded in this image?") == NONFLOODED
assert fn_attr("How many flooded buildings are there?") == FLOODED and fn_object("How many flooded buildings?") == "building"
assert fn_yesno_parts("Is the entire road non flooded?") == dict(obj="road", polarity=NONFLOODED, quant="entire")
assert fn_yesno_parts("Is the entire road flooded?")["polarity"] == FLOODED
assert parse_condition("non-flooded") == NONFLOODED and parse_condition("Flooded.") == FLOODED
assert parse_condition("partially flooded") == "mixed" and parse_amount("Some of it") == "mixed"
assert normalize_answer("The road is partly flooded", "floodnet", "condition", "condition of the road?") == MIXED
assert normalize_answer("partly flooded", "floodnet", "condition", "overall condition of the image?") == FLOODED
assert gt_label("flooded, non-flooded", "floodnet", "condition") == MIXED and gt_label(7, "floodnet", "count_simple") == "7"
assert normalize_answer("There are 12 buildings.", "floodnet", "count_simple") == "12"
assert parse_condition("Good") == NONFLOODED and parse_condition("Waterlogged") == FLOODED
assert parse_instance("Dry.") == NONFLOODED and parse_instance("yes") == FLOODED
assert parse_yesno("present") == "yes" and parse_yesno("absent") == "no"
assert gt_label("17", "rsvqa_lr", "count") == "11-100"
print("question parser self-tests passed ✓")

## 4 · Datasets
* **FloodNet-VQA**: the Hugging Face mirror of the official challenge data (`takara-ai/FloodNet_2021-Track_2_Dataset_HF`). Only the annotations (~1,450 small JSON files) and the **sampled** images (~5 MB each) are downloaded, not the 12 GB repository.
* **RSVQA-LR**: downloaded from Zenodo (record 6344334).

In [ ]:
# ── Datasets: RSVQA-LR (Zenodo) · FloodNet-VQA (Hugging Face mirror) · synthetic stand-ins ──
ZENODO = "https://zenodo.org/records/6344334/files/{}?download=1"

def _download(url, dest):
    if dest.exists() and dest.stat().st_size > 0: return
    tmp = dest.with_suffix(dest.suffix + ".part"); print("  downloading", dest.name, "…", flush=True)
    urllib.request.urlretrieve(url, tmp); tmp.rename(dest)

# ---------------- RSVQA-LR ----------------
def download_rsvqa_lr():
    d = data_dir("rsvqa_lr")
    for s in ("train", "val", "test"):
        for k in ("questions", "answers", "images"):
            _download(ZENODO.format(f"LR_split_{s}_{k}.json"), d / f"LR_split_{s}_{k}.json")
    _download(ZENODO.format("Images_LR.zip"), d / "Images_LR.zip")
    if not any((d / "Images_LR").glob("*.tif")):
        with zipfile.ZipFile(d / "Images_LR.zip") as z: z.extractall(d)

def load_rsvqa_lr():
    d, rows = data_dir("rsvqa_lr"), []
    for split in ("train", "val", "test"):
        Q = json.loads((d / f"LR_split_{split}_questions.json").read_text())["questions"]
        A = {a["id"]: a for a in json.loads((d / f"LR_split_{split}_answers.json").read_text())["answers"]
             if a.get("active", True)}
        for q in Q:
            if not q.get("active", True) or not q.get("answers_ids"): continue
            a = A.get(q["answers_ids"][0])
            if a is None: continue
            rows.append(dict(qid=f"lr:{q['id']}", img=str(q["img_id"]), split=split, type=q["type"],
                             question=q["question"], answer_raw=str(a["answer"])))
    return pd.DataFrame(rows)

# ---------------- FloodNet-VQA ----------------
def download_floodnet_annotations():
    d = data_dir("floodnet")
    if len(list((d / "train_image" / "ann").glob("*.json"))) > 1000: return
    from huggingface_hub import snapshot_download
    print("  downloading FloodNet-VQA annotations (≈1,450 small JSON files) …")
    snapshot_download(FLOODNET_HF, repo_type="dataset", allow_patterns=["train_image/ann/*"], local_dir=str(d))

def fetch_floodnet_images(names):
    """Download only the sampled images (≈5 MB each) instead of the 12 GB repository."""
    d = data_dir("floodnet")
    missing = [n for n in names if not (d / "train_image" / "img" / n).exists()]
    if not missing: return
    from huggingface_hub import hf_hub_download
    for n in tqdm(missing, desc="FloodNet images"):
        hf_hub_download(FLOODNET_HF, filename=f"train_image/img/{n}", repo_type="dataset", local_dir=str(d))

def _parse_tag(v):
    if isinstance(v, dict): return v
    try: return ast.literal_eval(v)
    except (ValueError, SyntaxError):
        return json.loads(v)

def load_floodnet():
    d, rows = data_dir("floodnet"), []
    for p in sorted((d / "train_image" / "ann").glob("*.json")):
        ann = json.loads(p.read_text()); img = p.name[:-len(".json")]
        for tag in ann.get("tags", []):
            if tag.get("name") != "question": continue
            v = _parse_tag(tag["value"])
            rows.append(dict(qid=f"fn:{v['Question_ID']}", img=img, split="train",
                             type=FLOODNET_TYPEMAP.get(v["Question_Type"], v["Question_Type"]),
                             question=v["Question"], answer_raw=str(v["Ground_Truth"]),
                             W=ann.get("size", {}).get("width"), H=ann.get("size", {}).get("height")))
    df = pd.DataFrame(rows)
    if df.qid.duplicated().any():                                  # make ids unique if the mirror repeats them
        df["qid"] = [f"fn:{i}" for i in range(len(df))]
    return df

# ---------------- synthetic stand-ins (DRY_RUN) ----------------
SYN_LR_CLASSES = {"residential building": (200, 70, 60), "commercial building": (235, 150, 60),
                  "water area": (40, 90, 170), "road": (150, 150, 150)}
SYN_PLURAL = {"residential building": "residential buildings", "commercial building": "commercial buildings",
              "water area": "water areas", "road": "roads", "building": "buildings"}

def _lr_size_class(area): return "small" if area < 30 else ("medium" if area < 100 else "large")
def syn_match(o, phrase):
    phrase = phrase.replace("buildings", "building").replace("areas", "area").replace("roads", "road")
    size = SIZE_RE.search(phrase)
    if size and _lr_size_class(o["area"]) != size.group(1): return False
    core = SIZE_RE.sub("", phrase).strip()
    return core == o["cls"] or (core == "building" and o["cls"].endswith("building"))

def build_synthetic_lr(n={"train": 24, "val": 12, "test": 22}):
    d, rng = data_dir("rsvqa_lr"), np.random.default_rng(SEED)
    (d / "Images_LR").mkdir(exist_ok=True); scenes, img_id, qid = {}, 0, 0
    nouns = ["building", "residential building", "commercial building", "water area", "road"]
    for split, k in n.items():
        Q, A = [], []
        for _ in range(k):
            urban = bool(rng.random() < 0.5); objs = []
            def add(cls, m, a, b):
                for _ in range(m):
                    w, h = rng.integers(a, b + 1), rng.integers(a, b + 1); x, y = rng.integers(0, 256 - w), rng.integers(0, 256 - h)
                    objs.append(dict(cls=cls, box=[int(x), int(y), int(x + w), int(y + h)], area=int(w * h)))
            add("residential building", int(rng.integers(15, 60) if urban else rng.integers(0, 12)), 2, 9)
            add("commercial building", int(rng.integers(3, 15) if urban else rng.integers(0, 3)), 4, 14)
            add("water area", int(rng.integers(0, 4)), 8, 36)
            add("road", int(rng.integers(3, 10) if urban else rng.integers(1, 4)), 3, 3)
            scenes[str(img_id)] = dict(urban=urban, objects=objs, W=256, H=256)
            img = np.array([120, 120, 115] if urban else [95, 125, 70], np.float32) + rng.normal(0, 9, (256, 256, 3))
            for o in objs:
                x0, y0, x1, y1 = o["box"]; img[y0:y1, x0:x1] = SYN_LR_CLASSES[o["cls"]]
            Image.fromarray(np.clip(img, 0, 255).astype(np.uint8)).save(d / "Images_LR" / f"{img_id}.tif")
            qs = []
            for _ in range(3):
                adj = rng.choice(["", "", "small ", "large "]); ph = f"{adj}{rng.choice(nouns)}".strip()
                if rng.random() < 0.3:
                    other = str(rng.choice(nouns)); q = f"Is there a {ph} next to a {other}?"
                    ans = any(syn_match(o, ph) for o in objs) and any(syn_match(o, other) for o in objs) and rng.random() < 0.6
                else:
                    q = rng.choice([f"Is there a {ph}?", f"Is a {ph} present?"]); ans = any(syn_match(o, ph) for o in objs)
                qs.append(("presence", q, "yes" if ans else "no"))
                pl = (rng.choice(["", "small "]) + SYN_PLURAL[str(rng.choice(nouns))]).strip()
                qs.append(("count", f"How many {pl} are there?", str(sum(syn_match(o, pl) for o in objs))))
                a_, b_ = rng.choice(nouns, 2, replace=False)
                ca, cb = sum(syn_match(o, a_) for o in objs), sum(syn_match(o, b_) for o in objs)
                f = rng.integers(0, 3)
                q, ans = [(f"Are there more {SYN_PLURAL[a_]} than {SYN_PLURAL[b_]}?", ca > cb),
                          (f"Are there less {SYN_PLURAL[a_]} than {SYN_PLURAL[b_]}?", ca < cb),
                          (f"Is the number of {SYN_PLURAL[a_]} equal to the number of {SYN_PLURAL[b_]}?", ca == cb)][f]
                qs.append(("comp", q, "yes" if ans else "no"))
            qs.append(("rural_urban", "Is it a rural or an urban area?", "urban" if urban else "rural"))
            for t, q, a in qs:
                Q.append(dict(id=qid, img_id=img_id, type=t, question=q, answers_ids=[qid], active=True))
                A.append(dict(id=qid, question_id=qid, answer=a, active=True)); qid += 1
            img_id += 1
        (d / f"LR_split_{split}_questions.json").write_text(json.dumps({"questions": Q}))
        (d / f"LR_split_{split}_answers.json").write_text(json.dumps({"answers": A}))
    (d / "scenes.json").write_text(json.dumps(scenes))

def build_synthetic_floodnet(n_images=110, W=1600, H=1200):
    """Small UAV-like scenes in the exact Supervisely/HF format of the real mirror."""
    d, rng = data_dir("floodnet"), np.random.default_rng(SEED + 7)
    (d / "train_image" / "ann").mkdir(parents=True, exist_ok=True); (d / "train_image" / "img").mkdir(parents=True, exist_ok=True)
    scenes, qid = {}, 0
    for i in range(n_images):
        name = f"{7000 + i}.JPG"; flooded_scene = rng.random() < 0.55
        water = []
        if flooded_scene:
            for _ in range(rng.integers(1, 4)):
                w, h = rng.integers(300, 900), rng.integers(250, 700)
                x, y = rng.integers(0, W - w), rng.integers(0, H - h); water.append([int(x), int(y), int(x + w), int(y + h)])
        def wet(b):
            return any(not (b[2] < w[0] or b[0] > w[2] or b[3] < w[1] or b[1] > w[3]) for w in water)
        roads = []
        if rng.random() < 0.8:
            y = int(rng.integers(150, H - 150)); roads.append([0, y, W, y + 34])
            if rng.random() < 0.5:
                x = int(rng.integers(150, W - 150)); roads.append([x, 0, x + 34, H])
        road_segs = []
        for r in roads:                                              # split roads into segments for flood state
            if r[2] - r[0] > r[3] - r[1]:
                for x in range(0, W, 200): road_segs.append([x, r[1], min(W, x + 200), r[3]])
            else:
                for y in range(0, H, 200): road_segs.append([r[0], y, r[2], min(H, y + 200)])
        bld = []
        for _ in range(int(rng.integers(0, 22))):
            s = int(rng.integers(22, 70)); x, y = int(rng.integers(0, W - s)), int(rng.integers(0, H - s))
            bld.append(dict(box=[x, y, x + s, y + s], area=s * s))
        for b in bld: b["flooded"] = wet(b["box"])
        segs = [dict(box=s, flooded=wet(s)) for s in road_segs]
        scenes[name] = dict(W=W, H=H, water=water, buildings=bld, roads=segs)
        img = np.zeros((H, W, 3), np.float32) + np.array([86, 118, 70]) + rng.normal(0, 8, (H, W, 3))
        for s in segs:
            x0, y0, x1, y1 = s["box"]; img[y0:y1, x0:x1] = (140, 138, 132)
        for w in water:
            x0, y0, x1, y1 = w; img[y0:y1, x0:x1] = img[y0:y1, x0:x1] * 0.35 + np.array([110, 96, 70]) * 0.65
        for b in bld:
            x0, y0, x1, y1 = b["box"]; img[y0:y1, x0:x1] = (178, 70, 60) if not b["flooded"] else (150, 85, 70)
        Image.fromarray(np.clip(img, 0, 255).astype(np.uint8)).save(d / "train_image" / "img" / name, quality=88)
        nb_f = sum(b["flooded"] for b in bld); nb = len(bld)
        qs = [("What is the overall condition of the given image?", FLOODED if water else "non flooded", "Condition_Recognition")]
        if segs:
            fl = {s["flooded"] for s in segs}
            rc = MIXED if len(fl) == 2 else (FLOODED if True in fl else "non flooded")
            qs.append(("What is the condition of the road in this image?", rc.replace(",", ", ") if rng.random() < 0.3 else rc,
                       "Condition_Recognition"))
            if rng.random() < 0.5: qs.append(("Is the entire road flooded?", "Yes" if fl == {True} else "No", "Yes_No"))
            else:                  qs.append(("Is the entire road non flooded?", "Yes" if fl == {False} else "No", "Yes_No"))
        qs.append(("How many buildings are in the image?", nb, "Simple_Counting"))
        if rng.random() < 0.5: qs.append(("How many flooded buildings are in the image?", nb_f, "Complex_Counting"))
        else:                  qs.append(("How many buildings are non flooded in this image?", nb - nb_f, "Complex_Counting"))
        tags = []
        for q, a, t in qs:
            tags.append(dict(id=qid, tagId=1, name="question",
                             value=repr({"Question_ID": str(qid), "Question": q, "Ground_Truth": a, "Question_Type": t})))
            qid += 1
        (d / "train_image" / "ann" / f"{name}.json").write_text(json.dumps(
            dict(description="", tags=tags, size=dict(height=H, width=W), objects=[])))
    (d / "scenes.json").write_text(json.dumps(scenes))

ALLQ = {}
for ds in DATASETS:
    if "data" in STAGES:
        if DRY_RUN:
            if not (data_dir(ds) / "scenes.json").exists():
                (build_synthetic_lr if ds == "rsvqa_lr" else build_synthetic_floodnet)()
        else:
            (download_rsvqa_lr if ds == "rsvqa_lr" else download_floodnet_annotations)()
    df = load_rsvqa_lr() if ds == "rsvqa_lr" else load_floodnet()
    df["ds"] = ds
    df["label"] = [gt_label(a, ds, t) for a, t in zip(df.answer_raw, df.type)]
    df = df[df.label.notna() & df.type.isin(TYPES[ds])]
    ALLQ[ds] = df.reset_index(drop=True)
    print(f"{DS_LABEL[ds]:28s}: {df.img.nunique():6d} images {len(df):7d} questions | "
          + ", ".join(f"{k}={v}" for k, v in df.type.value_counts().items()))
if "floodnet" in ALLQ:
    print("FloodNet condition labels:", dict(ALLQ["floodnet"][ALLQ["floodnet"].type == "condition"].label.value_counts()))

### Sampling
* **FloodNet-VQA:** the answers of the official val/test splits were never released, so (as in prior work) DEV and EVAL images are drawn **disjointly from the 1,448 annotated training images**. Every question of a sampled image is used.
* **RSVQA-LR:** the **same sample as run 1** (40 train images for DEV, 160 test/val images for EVAL), so the two runs are directly comparable.
* DEV is used only for reliability weights and thresholds. Every reported number comes from EVAL. The most-common-answer baseline uses training questions that are in neither sample.

In [ ]:
def _pick_lr(g, rng, slot4_rural):            # identical to run 1
    chosen = []
    for t in ("count", "presence", "comp"):
        c = g[g.type == t]
        if len(c): chosen.append(c.sample(1, random_state=int(rng.integers(1 << 30))).index[0])
    rest = g.drop(index=chosen)
    if slot4_rural and (rest.type == "rural_urban").any():
        chosen.append(rest[rest.type == "rural_urban"].index[0])
    else:
        pool = rest[rest.type.isin(["presence", "count"])]
        tagged = pool[[(SMALL_RE.search(_norm(q)) or REL_RE.search(_norm(q))) is not None for q in pool.question]]
        pool = tagged if len(tagged) else (pool if len(pool) else rest[rest.type != "rural_urban"])
        if len(pool): chosen.append(pool.sample(1, random_state=int(rng.integers(1 << 30))).index[0])
    return g.loc[chosen[:4]]

def _draw_lr(df_list, n, rng, role):
    df = pd.concat(df_list)
    ok = df.groupby("img").type.apply(lambda s: {"count", "presence", "comp"} <= set(s))
    imgs = [i for i in df.img.unique() if ok.get(i, False)]; rng.shuffle(imgs)
    first = df.drop_duplicates("img").set_index("img").split
    imgs = sorted(imgs, key=lambda i: 0 if first[i] == df_list[0].split.iloc[0] else 1)[:n]
    return pd.concat([_pick_lr(df[df.img == i], rng, k % 2 == 0) for k, i in enumerate(imgs)]).assign(role=role)

def sample_lr():
    v1 = Path(PROJECT_ROOT) / "outputs" / "sample_questions.csv"
    A = ALLQ["rsvqa_lr"]
    if v1.exists() and not DRY_RUN:                                   # reuse run-1 sample verbatim
        s = pd.read_csv(v1, dtype={"label": str, "answer_raw": str})
        s = pd.DataFrame(dict(qid="lr:" + s.qid.astype(str), role=s.role))
        print("RSVQA-LR: re-using the run-1 sample")
        return A.merge(s, on="qid")
    n = SAMPLE_SIZES["rsvqa_lr"]
    dev = _draw_lr([A[A.split == "train"]], n["dev"], np.random.default_rng(SEED + 1), "dev")
    ev = _draw_lr([A[A.split == "test"], A[A.split == "val"]], n["eval"], np.random.default_rng(SEED + 2), "eval")
    return pd.concat([dev, ev], ignore_index=True)

def sample_fn():
    A = ALLQ["floodnet"]; n = SAMPLE_SIZES["floodnet"]
    imgs = sorted(A.img.unique()); rng = np.random.default_rng(SEED + 3); rng.shuffle(imgs)
    if DRY_RUN: n = dict(dev=min(n["dev"], 30), eval=min(n["eval"], 60))
    dev, ev = set(imgs[:n["dev"]]), set(imgs[n["dev"]:n["dev"] + n["eval"]])
    s = A[A.img.isin(dev | ev)].copy(); s["role"] = np.where(s.img.isin(dev), "dev", "eval")
    return s

SAMPLE_PATH = OUT_DIR / "sample_v2.csv"
meta = dict(sizes=SAMPLE_SIZES, seed=SEED, dry=DRY_RUN, ds=sorted(DATASETS))
mp = OUT_DIR / "sample_v2_meta.json"
if SAMPLE_PATH.exists() and mp.exists() and json.loads(mp.read_text()) == json.loads(json.dumps(meta)):
    SAMPLE = pd.read_csv(SAMPLE_PATH, dtype={"label": str, "answer_raw": str, "img": str, "qid": str})
    print("loaded saved sample →", SAMPLE_PATH)
else:
    parts = [sample_lr() if ds == "rsvqa_lr" else sample_fn() for ds in DATASETS]
    SAMPLE = pd.concat(parts, ignore_index=True)
    SAMPLE.to_csv(SAMPLE_PATH, index=False); mp.write_text(json.dumps(meta))
    print("new sample saved →", SAMPLE_PATH)

if "floodnet" in DATASETS and not DRY_RUN and ("data" in STAGES or "run" in STAGES):
    fetch_floodnet_images(sorted(SAMPLE[SAMPLE.ds == "floodnet"].img.unique()))

SAMPLE["route"] = [route(q, ds) for q, ds in zip(SAMPLE.question, SAMPLE.ds)]
SAMPLE["kind"] = [kind_of(ds, t) for ds, t in zip(SAMPLE.ds, SAMPLE.type)]
tg = pd.DataFrame([question_tags(q, ds) for q, ds in zip(SAMPLE.question, SAMPLE.ds)])
SAMPLE[["small", "relational", "subgroup"]] = tg[["small", "relational", "subgroup"]].values
SAMPLE["img_key"] = SAMPLE.ds + ":" + SAMPLE.img.astype(str)
for ds in DATASETS:
    s = SAMPLE[SAMPLE.ds == ds]; acc = (s.route == s.type).mean()
    print(f"{DS_SHORT[ds]}: router agreement with dataset types {acc:.1%}")
    if acc < 0.98:
        display(s[s.route != s.type][["question", "type", "route"]].drop_duplicates("question").head(12))

# most-common answer per type, from training questions that are in neither sample
MAJORITY = {}
for ds in DATASETS:
    A = ALLQ[ds]
    pool = A[A.split == "train"] if ds == "rsvqa_lr" else A[~A.img.isin(SAMPLE[SAMPLE.ds == ds].img)]
    if pool.empty: pool = A
    for t, g in pool.groupby("type"):
        MAJORITY[(ds, t)] = g.label.value_counts().index[0]
# label space of condition questions = answers that occur in ≥ 5 % of the training pool (per sub-type)
if "floodnet" in DATASETS:
    A = ALLQ["floodnet"]; pool = A[~A.img.isin(SAMPLE[SAMPLE.ds == "floodnet"].img) & (A.type == "condition")]
    for sub in ("road", "image"):
        g = pool[[cond_subtype(q) == sub for q in pool.question]]
        if len(g):
            f = g.label.value_counts(normalize=True)
            COND_LABELS[sub] = [l for l in (MIXED, FLOODED, NONFLOODED) if f.get(l, 0) >= 0.05]
            print(f"condition/{sub}: training answer mix {f.round(3).to_dict()} → label space {COND_LABELS[sub]}")
print("most-common answers:", {f"{k[0]}/{k[1]}": v for k, v in MAJORITY.items()})

ROBUST_IMGS = {ds: list(SAMPLE[(SAMPLE.ds == ds) & (SAMPLE.role == "eval")].img.drop_duplicates()
                        [: (10 if DRY_RUN else ROBUST_N_IMAGES)]) for ds in ROBUST_DATASETS if ds in DATASETS}
display(SAMPLE.groupby(["ds", "role", "type"]).size().unstack(fill_value=0)
        .assign(images=SAMPLE.groupby(["ds", "role"]).img.nunique(), total=SAMPLE.groupby(["ds", "role"]).size()))

## 5 · Images, native tiles, resolution gate, UAV perturbations

In [ ]:
# ── Images, native tiles, resolution gate, UAV perturbations ──────────────────
try:
    import tifffile
except ImportError:
    tifffile = None
import io

class _LRU(OrderedDict):
    def __init__(self, n): super().__init__(); self.n = n
    def get_or(self, k, fn):
        if k in self: self.move_to_end(k); return self[k]
        v = fn(); self[k] = v
        if len(self) > self.n: self.popitem(last=False)
        return v
_RAW, _PERT = _LRU(6), _LRU(4)

def _read(ds, img):
    if ds == "rsvqa_lr":
        p = data_dir(ds) / "Images_LR" / f"{img}.tif"
        if not p.exists(): p = next(data_dir(ds).rglob(f"{img}.tif"))
        arr = tifffile.imread(p) if tifffile is not None else np.array(Image.open(p))
        if arr.ndim == 3 and arr.shape[0] in (3, 4) and arr.shape[-1] not in (3, 4): arr = np.moveaxis(arr, 0, -1)
        arr = arr[..., :3]
        if arr.dtype != np.uint8:
            lo, hi = np.percentile(arr, (2, 98)); arr = np.clip((arr - lo) / max(hi - lo, 1e-6) * 255, 0, 255)
        return Image.fromarray(arr.astype(np.uint8)).convert("RGB")
    return Image.open(data_dir(ds) / "train_image" / "img" / img).convert("RGB")

def load_image(ds, img): return _RAW.get_or((ds, img), lambda: _read(ds, img))

def _motion_blur(im, L):
    a = np.asarray(im, np.float32); L = int(max(1, L)); pad = L // 2
    c = np.cumsum(np.pad(a, ((0, 0), (pad + 1, L - pad - 1), (0, 0)), mode="edge"), axis=1)
    out = (c[:, L:, :] - c[:, :-L, :]) / L
    return Image.fromarray(np.clip(out, 0, 255).astype(np.uint8))

def perturb(im, kind, level):
    if kind in (None, "clean"): return im
    w, h = im.size
    if kind == "altitude":
        return im.resize((max(8, int(w * level)), max(8, int(h * level))), Image.BICUBIC).resize((w, h), Image.BICUBIC)
    if kind == "motion_blur": return _motion_blur(im, level * w / 4000)     # length defined at 4000-px width
    if kind == "haze":
        a = np.asarray(im, np.float32); A = np.array([214, 216, 220], np.float32)
        return Image.fromarray(np.clip(a * (1 - level) + A * level, 0, 255).astype(np.uint8))
    if kind == "jpeg":
        b = io.BytesIO(); im.save(b, "JPEG", quality=int(level)); b.seek(0); return Image.open(b).convert("RGB")
    raise ValueError(kind)

def get_image(ds, img, pert="clean", level=0.0):
    return _PERT.get_or((ds, img, pert, level), lambda: perturb(load_image(ds, img), pert, level))

@dataclass
class View:
    pil: Image.Image
    key: str
    ds: str = ""
    img: str = ""
    box: tuple = (0, 0, 1, 1)       # region in native pixel coordinates
    native: tuple = (1, 1)          # native size of the region (px)
    pert: str = "clean"
    level: float = 0.0

def full_view(ds, img, pert="clean", level=0.0):
    im = get_image(ds, img, pert, level)
    return View(im, f"{ds}|{img}|{pert}{level}|full", ds, img, (0, 0) + im.size, im.size, pert, level)

def tile_boxes(size, grid=GRID):
    w, h = size
    return {f"r{r}c{c}": (c * w // grid, r * h // grid, (c + 1) * w // grid, (r + 1) * h // grid)
            for r in range(grid) for c in range(grid)}

def zoom_gain(size, grid=GRID):
    """Native pixels per model pixel inside one tile. >1 → tiles reveal detail the full view cannot."""
    return (max(size) / grid) / MODEL_SIDE

def tile_views(ds, img, pert="clean", level=0.0, upscale_to=None):
    im = get_image(ds, img, pert, level); out = {}
    for n, b in tile_boxes(im.size).items():
        c = im.crop(b)
        if upscale_to: c = c.resize((upscale_to, upscale_to), Image.LANCZOS)
        out[n] = View(c, f"{ds}|{img}|{pert}{level}|{n}" + (f"@{upscale_to}" if upscale_to else ""),
                      ds, img, b, (b[2] - b[0], b[3] - b[1]), pert, level)
    return out

def box_view(fv, box, k, margin=0.35, min_side=96):
    """Crop one detected instance (native coords) with context margin."""
    W, H = fv.pil.size; x0, y0, x1, y1 = box
    mw, mh = max((x1 - x0) * margin, (min_side - (x1 - x0)) / 2, 0), max((y1 - y0) * margin, (min_side - (y1 - y0)) / 2, 0)
    b = (int(max(0, x0 - mw)), int(max(0, y0 - mh)), int(min(W, x1 + mw)), int(min(H, y1 + mh)))
    return View(fv.pil.crop(b), f"{fv.key}|box{k}:{b}", fv.ds, fv.img, b, (b[2] - b[0], b[3] - b[1]), fv.pert, fv.level)

BLANK_VIEW = View(Image.new("RGB", (512, 512), (128, 128, 128)), "blank", "", "", (0, 0, 512, 512), (512, 512))

## 6 · Models
Answerers: **Moondream2** (pinned `2025-06-21`) and **SmolVLM-Instruct**, both with greedy decoding. One **shared grounding module** (the Moondream2 detect head) serves both, so the comparison between answerers is fair. Every call is cached with its latency and peak memory.

In [ ]:
# ── VLM answerers, shared grounding module, persistent call cache ─────────────
def _infer(fn):
    return torch.inference_mode()(fn) if torch is not None else fn

class CallCache:
    """One JSON line per model call (prompt → raw output, latency, peak memory). Resumable."""
    def __init__(self, name):
        self.path = CACHE_DIR / f"calls_{name}.jsonl"; self.db = {}
        if self.path.exists():
            with open(self.path) as f:
                for line in f:
                    try: r = json.loads(line); self.db[r["k"]] = r
                    except json.JSONDecodeError: pass
        self._fh = open(self.path, "a"); self.hits = self.misses = 0
    @staticmethod
    def key(*p): return hashlib.sha1("§".join(map(str, p)).encode()).hexdigest()[:20]
    def get(self, k):
        r = self.db.get(k)
        if r is not None: self.hits += 1
        return r
    def put(self, k, rec):
        rec = dict(rec, k=k); self.db[k] = rec; self.misses += 1
        self._fh.write(json.dumps(rec) + "\n"); self._fh.flush(); return rec

def _timed(fn):
    if HAS_CUDA: torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter(); out = fn()
    if HAS_CUDA: torch.cuda.synchronize()
    return out, time.perf_counter() - t0, (torch.cuda.max_memory_allocated() / 2**20 if HAS_CUDA else float("nan"))

class Answerer:
    name = "base"
    def __init__(self, cache): self.cache = cache
    def timed(self, fn): return _timed(fn)
    def ask(self, view, prompt, intent=None):
        k = CallCache.key(self.name, "query", view.key, prompt)
        r = self.cache.get(k)
        if r is None:
            out, lat, mem = self.timed(lambda: self._query(view, prompt, intent))
            r = self.cache.put(k, dict(kind="query", view=view.key, prompt=prompt, out=out, lat=lat, mem=mem))
        return r
    def static_memory_mb(self): return torch.cuda.memory_allocated() / 2**20 if HAS_CUDA else float("nan")

class Grounder:
    """Open-vocabulary grounding shared by every answerer (Moondream2 detect head)."""
    name = "grounder"
    def __init__(self, cache, model=None): self.cache, self.model = cache, model
    def timed(self, fn): return _timed(fn)
    def detect(self, view, phrase):
        k = CallCache.key(self.name, "detect", view.key, phrase, DET_MAX_OBJECTS)
        r = self.cache.get(k)
        if r is None:
            boxes, lat, mem = self.timed(lambda: self._detect(view, phrase))
            r = self.cache.put(k, dict(kind="detect", view=view.key, prompt=phrase, out=boxes, lat=lat, mem=mem))
        return r
    @_infer
    def _detect(self, view, phrase):
        objs = self.model.detect(view.pil, phrase, settings={"max_objects": DET_MAX_OBJECTS, "variant": None})["objects"]
        x0, y0 = view.box[:2]; w, h = view.native
        return [[x0 + o["x_min"] * w, y0 + o["y_min"] * h, x0 + o["x_max"] * w, y0 + o["y_max"] * h] for o in objs]

def _load_moondream(spec):
    from transformers import AutoModelForCausalLM
    return AutoModelForCausalLM.from_pretrained(spec["hf_id"], revision=spec["revision"], trust_remote_code=True,
                                                torch_dtype=torch.bfloat16, device_map={"": "cuda"}).eval()

class Moondream2(Answerer):
    name = "moondream2"
    def __init__(self, cache, spec):
        super().__init__(cache); self.model = _load_moondream(spec)
        self.settings = {"temperature": 0.0, "top_p": 0.3, "max_tokens": MAX_NEW_TOKENS, "variant": None}
    @_infer
    def _query(self, view, prompt, intent=None):
        torch.manual_seed(SEED)
        try:
            out = self.model.query(view.pil, prompt, settings=self.settings)
        except (RuntimeError, ZeroDivisionError, ValueError):
            if self.settings["temperature"] != 0.0: raise
            self.settings["temperature"] = 1e-4; out = self.model.query(view.pil, prompt, settings=self.settings)
        return str(out["answer"]).strip()

class SmolVLM(Answerer):
    name = "smolvlm"
    def __init__(self, cache, spec):
        super().__init__(cache)
        from transformers import AutoProcessor
        try:
            from transformers import AutoModelForImageTextToText as AutoVLM
        except ImportError:
            from transformers import AutoModelForVision2Seq as AutoVLM
        self.processor = AutoProcessor.from_pretrained(spec["hf_id"], revision=spec["revision"],
                                                       size={"longest_edge": spec["longest_edge"]})
        self.model = AutoVLM.from_pretrained(spec["hf_id"], revision=spec["revision"], torch_dtype=torch.bfloat16,
                                             _attn_implementation="sdpa").to("cuda").eval()
    @_infer
    def _query(self, view, prompt, intent=None):
        msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
        text = self.processor.apply_chat_template(msgs, add_generation_prompt=True)
        inp = self.processor(text=text, images=[view.pil], return_tensors="pt")
        inp = {k: (v.to("cuda", dtype=torch.bfloat16) if torch.is_floating_point(v) else v.to("cuda")) for k, v in inp.items()}
        out = self.model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        return self.processor.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

# ---------------- mock models for DRY_RUN (simulate run-1 behaviour, incl. yes-bias) ----------------
class _MockWorld:
    def __init__(self):
        self.sc = {ds: json.loads((data_dir(ds) / "scenes.json").read_text()) for ds in DATASETS}
    def scene(self, v): return self.sc[v.ds][str(v.img)]
    @staticmethod
    def quality(v):
        q = 1.0
        if v.pert == "altitude": q *= v.level ** 0.8
        if v.pert == "motion_blur": q *= math.exp(-v.level / 60)
        if v.pert == "haze": q *= (1 - v.level) ** 0.8
        if v.pert == "jpeg": q *= min(1, 0.55 + v.level / 60)
        return q
    def eff_area(self, v, area):
        s = min(1.0, MODEL_SIDE / max(v.native)) * self.quality(v)
        return area * s * s
    def inside(self, v, box):
        cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
        return v.box[0] <= cx < v.box[2] and v.box[1] <= cy < v.box[3]
    def seen(self, v, area, rng, a0=250):
        e = self.eff_area(v, area)
        return rng.random() < 1 / (1 + math.exp(-(math.log(e + 1e-6) - math.log(a0)) * 1.6))
    def objects(self, v, phrase, rng, attr=None):
        if v.key == "blank": return []
        s = self.scene(v); out = []
        if v.ds == "rsvqa_lr":
            for o in s["objects"]:
                if self.inside(v, o["box"]) and syn_match(o, phrase) and self.seen(v, o["area"], rng, a0=60): out.append(o)
            return out
        p = phrase.lower()
        pool = s["buildings"] if ("build" in p or "house" in p or "roof" in p) else (s["roads"] if "road" in p else [])
        for o in pool:
            area = o.get("area") or (o["box"][2] - o["box"][0]) * (o["box"][3] - o["box"][1])
            if not self.inside(v, o["box"]) or not self.seen(v, area, rng): continue
            if attr is not None:
                fl = o["flooded"] if rng.random() < 0.85 * self.quality(v) + 0.1 else not o["flooded"]
                if (attr == FLOODED) != fl: continue
            out.append(o)
        return out

class MockAnswerer(Answerer):
    def __init__(self, cache, name, skill, yes_bias):
        super().__init__(cache); self.name, self.skill, self.yes_bias = name, skill, yes_bias; self.W = _MockWorld()
    def timed(self, fn):
        out = fn(); r = np.random.default_rng(len(self.cache.db))
        return out, (0.09 if self.name == "moondream2" else 0.14) * r.uniform(0.85, 1.25), self.static_memory_mb() + r.uniform(200, 600)
    def static_memory_mb(self): return {"moondream2": 3680.0, "smolvlm": 4300.0}.get(self.name, 4000.0)
    def _rng(self, v, p): return np.random.default_rng(int(CallCache.key(self.name, v.key, p), 16) % 2**32)
    def _yn(self, truth, rng, style):
        bias = self.yes_bias if style == "yesno" else self.yes_bias * 0.25
        a = truth
        if not truth and rng.random() < bias: a = True
        if rng.random() < 0.08 / self.skill: a = not a
        if style == "forced": return rng.choice(["present", "Present."]) if a else rng.choice(["absent", "Absent."])
        if style == "truefalse": return "true" if a else "false"
        return rng.choice(["Yes.", "yes", "Yes, it is."]) if a else rng.choice(["No.", "no", "No, it is not."])
    def _query(self, v, prompt, it=None):
        rng, it = self._rng(v, prompt), (it or {"k": "free"}); k = it["k"]; W = self.W
        if k == "count":
            n = len(W.objects(v, it["target"], rng, it.get("attr")))
            n = max(0, int(round(n * rng.uniform(0.75, 1.15))))
            return rng.choice([f"{n}", f"There are {n}.", f"{n} {it['target']}"]) if rng.random() > 0.02 else "several"
        if k == "presence":
            truth = len(W.objects(v, it["target"], rng)) > 0
            return self._yn(truth, rng, it.get("style", "yesno"))
        if k == "comp":
            p = it["parts"]; na, nb = len(W.objects(v, p["a"], rng)), len(W.objects(v, p["b"], rng))
            t = na > nb if p["op"] == "more" else (na < nb if p["op"] == "less" else na == nb)
            return self._yn(t, rng, it.get("style", "yesno"))
        if k == "ru":
            urban = W.scene(v)["urban"] if v.key != "blank" else rng.random() < 0.4
            if rng.random() > 0.8 * W.quality(v) + 0.1: urban = not urban
            st = it.get("style", "word")
            if st == "city": return "city" if urban else "countryside"
            return ("urban" if urban else "rural") if rng.random() > 0.2 else f"It is a {'urban' if urban else 'rural'} area."
        if k in ("cond", "fn_yesno", "road_tile", "box_flooded"):
            if v.key == "blank":
                st = rng.choice([FLOODED, NONFLOODED, "mixed"])
            else:
                s = W.scene(v)
                if k == "box_flooded" or it.get("obj", "road") == "image":
                    pool = s["buildings"] if k == "box_flooded" else [dict(box=w, flooded=True) for w in s["water"]]
                    hit = [o for o in pool if W.inside(v, o["box"])]
                    if k == "box_flooded":
                        fl = bool(hit and hit[0]["flooded"])
                        if rng.random() > 0.85 * W.quality(v) + 0.1: fl = not fl
                        return ("flooded" if fl else "dry") if rng.random() > 0.15 else self._yn(fl, rng, "yesno")
                    st = FLOODED if hit else NONFLOODED
                else:
                    segs = [o for o in s["roads"] if W.inside(v, o["box"])]
                    if not segs: st = "noroad"
                    else:
                        fl = {o["flooded"] if rng.random() < 0.85 * W.quality(v) + 0.1 else not o["flooded"] for o in segs}
                        st = "mixed" if len(fl) == 2 else (FLOODED if True in fl else NONFLOODED)
                if rng.random() < 0.12: st = rng.choice([FLOODED, NONFLOODED, "mixed"])
            if k == "road_tile":
                return "no road" if st == "noroad" else {"mixed": "partially flooded"}.get(st, st)
            if st == "noroad": st = NONFLOODED
            if k == "fn_yesno":
                p = it["parts"]; truth = st == (FLOODED if p["polarity"] == FLOODED else NONFLOODED)
                return self._yn(truth, rng, it.get("style", "yesno"))
            if it.get("style") == "amount":
                return {FLOODED: "all", NONFLOODED: "none", "mixed": "some"}[st]
            return {"mixed": rng.choice(["partially flooded", "flooded, non flooded"])}.get(st, st)
        return rng.choice(["yes", "no", "3", "flooded"])

class MockGrounder(Grounder):
    name = "grounder"
    def __init__(self, cache): super().__init__(cache); self.W = _MockWorld()
    def timed(self, fn):
        out = fn(); r = np.random.default_rng(len(self.cache.db) + 7)
        return out, 0.21 * r.uniform(0.85, 1.2), 4200 + r.uniform(0, 300)
    def _detect(self, v, phrase):
        rng = np.random.default_rng(int(CallCache.key("det", v.key, phrase), 16) % 2**32)
        return [list(map(float, o["box"])) for o in self.W.objects(v, phrase, rng)]

def load_models(name):
    cache = CallCache(name)
    if DRY_RUN:
        a = MockAnswerer(cache, name, skill=1.0 if name == "moondream2" else 0.8,
                         yes_bias=0.6 if name == "moondream2" else 0.25)
        return a, MockGrounder(cache)
    spec = MODEL_SPECS[name]; print(f"loading answerer {spec['hf_id']} @ {spec['revision']} …")
    a = {"moondream2": Moondream2, "smolvlm": SmolVLM}[name](cache, spec)
    g = Grounder(cache, a.model if name == "moondream2" else _load_moondream(GROUNDER))
    return a, g

def free_models(a, g):
    a.cache._fh.close(); del a, g; gc.collect()
    if HAS_CUDA: torch.cuda.empty_cache()

## 7 · Methods and evidence sources
* **E1:** the question plus the standard short-answer suffix.
* **E2:** a type-specific prompt on the full image.
* **E3:** 2×2 tiles: native resolution on FloodNet; upsampled crops on RSVQA-LR (the run-1 baseline).
* **Probe:** forced-choice or decomposed questions, e.g. *"how much of the road is flooded: none / some / all"* or *"present / absent"*.
* **Grounding:** instances are detected per tile, and for attribute counts each instance is verified (*"flooded or dry?"*).

In [ ]:
# ── Methods: prompts and evidence sources ─────────────────────────────────────
E1_SUFFIX = "Answer the question using a single word or phrase."      # standard short-answer VQA prompt
CTX  = {"floodnet": "This is a nadir UAV (drone) image of a neighbourhood.",
        "rsvqa_lr": "This is an overhead remote-sensing image seen from above."}
TCTX = {"floodnet": "This is one zoomed-in region of a UAV (drone) image.",
        "rsvqa_lr": "This is a zoomed-in region of an overhead remote-sensing image."}
FMT = {"yesno": "Answer with only one word: yes or no.", "num": "Answer with a single number only.",
       "ru": "Answer with only one word: rural or urban."}

def _plural(w): return w if w.endswith("s") else w + "s"
def fn_count_target(q):
    a = fn_attr(q); o = _plural(fn_object(q))
    return f"{a} {o}" if a else o

def _rec(r): return dict(out=r["out"] if r["kind"] == "query" else len(r["out"]), lat=r["lat"], mem=r["mem"])
def _yes(c): return "yes" if c else "no"
def _cmp(na, nb, op): return _yes(na > nb if op == "more" else (na < nb if op == "less" else na == nb))

# ---------- E2: structured single prompt on the full image ----------
def prompt_structured(ds, q, t):
    c = CTX[ds]
    if ds == "rsvqa_lr":
        return {"presence": f"{c} Look carefully at the whole image, including small objects. Question: {q} {FMT['yesno']}",
                "comp": f"{c} Compare how many of each kind of object are visible. Question: {q} {FMT['yesno']}",
                "count": f"{c} Count every separate instance carefully. Question: {q} {FMT['num']}",
                "rural_urban": f"{c} Judge from the density of buildings and roads and from the land use. "
                               f"Question: Is this a rural or an urban area? {FMT['ru']}"}[t]
    if t.startswith("count"):
        return f"{c} Count every separate {fn_count_target(q)[:-1]} carefully, including partly visible ones. Question: {q} {FMT['num']}"
    if t == "condition":
        opts = "flooded, non flooded" + (", partially flooded" if MIXED in COND_LABELS[cond_subtype(q)] else "")
        return f"{c} Question: {q} Answer with exactly one of: {opts}."
    return f"{c} Question: {q} {FMT['yesno']}"

def intent_for(ds, q, t, style="yesno"):
    if ds == "rsvqa_lr":
        if t == "presence": return dict(k="presence", target=presence_target(q) or "building", style=style)
        if t == "count": return dict(k="count", target=count_target(q) or "building")
        if t == "comp": return dict(k="comp", parts=comp_parts(q) or dict(op="more", a="building", b="road"), style=style)
        return dict(k="ru")
    if t.startswith("count"): return dict(k="count", target=_plural(fn_object(q)), attr=fn_attr(q))
    if t == "condition": return dict(k="cond", obj=cond_subtype(q))
    return dict(k="fn_yesno", parts=fn_yesno_parts(q) or dict(obj="road", polarity=FLOODED, quant="entire"), style=style)

def src_full(A, ds, q, t, fv):
    r = A.ask(fv, prompt_structured(ds, q, t), intent_for(ds, q, t))
    return dict(ans=normalize_answer(r["out"], ds, t, q), raw=r["out"], calls=[_rec(r)])

# ---------- tiles (E3; in E4 only when the resolution gate is open) ----------
ROAD_TILE = "Look at the road in this region. Reply with exactly one of: no road, flooded, non flooded, partially flooded."
def _road_states(A, tv):
    calls, per = [], {}
    for n, v in tv.items():
        r = A.ask(v, f"{TCTX['floodnet']} {ROAD_TILE}", dict(k="road_tile")); calls.append(_rec(r))
        s = str(r["out"]).lower()
        per[n] = None if re.search(r"\bno road\b|\bnone\b|\bnot visible\b", s) else parse_condition(s)
    return per, calls

def _agg_states(states):
    st = {s for s in states if s}
    if not st: return None
    if "mixed" in st or {FLOODED, NONFLOODED} <= st: return "mixed"
    return st.pop()

def src_tiles(A, ds, q, t, tv):
    tc, calls, per = TCTX[ds], [], {}
    if ds == "rsvqa_lr":
        if t == "presence":
            for n, v in tv.items():
                r = A.ask(v, f"{tc} Question: {q} {FMT['yesno']}", intent_for(ds, q, t)); calls.append(_rec(r)); per[n] = parse_yesno(r["out"])
            val = [a for a in per.values() if a]; ans = _yes("yes" in val) if val else None
        elif t == "count":
            tg = count_target(q)
            for n, v in tv.items():
                r = A.ask(v, f"{tc} How many {tg} are visible in this region? {FMT['num']}", intent_for(ds, q, t))
                calls.append(_rec(r)); per[n] = parse_count(r["out"])
            val = [c for c in per.values() if c is not None]; ans = count_bin(sum(val)) if val else None
        elif t == "comp":
            p = comp_parts(q); sa = sb = 0; ok = False
            if p:
                for n, v in tv.items():
                    ra = A.ask(v, f"{tc} How many {p['a']} are visible in this region? {FMT['num']}", dict(k="count", target=p["a"]))
                    rb = A.ask(v, f"{tc} How many {p['b']} are visible in this region? {FMT['num']}", dict(k="count", target=p["b"]))
                    calls += [_rec(ra), _rec(rb)]; ca, cb = parse_count(ra["out"]), parse_count(rb["out"])
                    per[n] = [ca, cb]; ok |= (ca is not None or cb is not None); sa += ca or 0; sb += cb or 0
            ans = _cmp(sa, sb, p["op"]) if ok else None
        else:
            for n, v in tv.items():
                r = A.ask(v, f"{tc} Is this region rural or urban? {FMT['ru']}", dict(k="ru")); calls.append(_rec(r)); per[n] = parse_rural_urban(r["out"])
            val = [a for a in per.values() if a]; ans = None if not val else ("urban" if 2 * val.count("urban") >= len(val) else "rural")
        return dict(ans=ans, per=per, calls=calls)
    # FloodNet (native-resolution tiles)
    if t.startswith("count"):
        tg = fn_count_target(q)
        for n, v in tv.items():
            r = A.ask(v, f"{tc} How many {tg} are visible in this region? {FMT['num']}", intent_for(ds, q, t))
            calls.append(_rec(r)); per[n] = parse_count(r["out"])
        val = [c for c in per.values() if c is not None]; ans = str(sum(val)) if val else None
    elif t == "condition" and cond_subtype(q) == "image":
        for n, v in tv.items():
            r = A.ask(v, f"{tc} Is there flood water in this region? Reply with exactly one word: flooded or dry.", dict(k="cond", obj="image"))
            calls.append(_rec(r)); per[n] = parse_condition(r["out"])
        val = [a for a in per.values() if a]                            # majority of tiles (OR over-reports flooding)
        ans = None if not val else (FLOODED if 2 * sum(a in (FLOODED, "mixed") for a in val) >= len(val) else NONFLOODED)
    else:
        per, calls = _road_states(A, tv); agg = _agg_states(per.values())
        if t == "condition":
            if MIXED in COND_LABELS["road"]:
                ans = cond_to_label(agg, "road")
            else:                                                       # binary label space → majority of road tiles
                vals = [s_ for s_ in per.values() if s_]
                ans = None if not vals else (FLOODED if 2 * sum(v in (FLOODED, "mixed") for v in vals) >= len(vals) else NONFLOODED)
        else:
            p = fn_yesno_parts(q); vals = [s for s in per.values() if s]
            if p and vals:
                if p["quant"] == "entire": ans = _yes(all(s == p["polarity"] for s in vals))
                else: ans = _yes(any(s in (p["polarity"], "mixed") for s in vals))
            else: ans = None
    return dict(ans=ans, per=per, calls=calls)

# ---------- contrastive / decomposed probe (avoids yes/no tokens where possible) ----------
def _swap_comp(p):
    a, b, op = p["a"], p["b"], p["op"]
    return {"more": f"Are there fewer {b} than {a}?", "less": f"Are there more {b} than {a}?"}.get(
        op, f"Is the number of {b} equal to the number of {a}?")

def _amount_probe(A, fv, obj):
    what = "the road" if obj == "road" else "the scene"
    p = f"{CTX['floodnet']} How much of {what} is covered by flood water? Reply with exactly one word: none, some, or all."
    r = A.ask(fv, p, dict(k="cond", obj=obj, style="amount")); return parse_amount(r["out"]), r, p

def src_probe(A, ds, q, t, fv):
    c = CTX[ds]
    if ds == "rsvqa_lr":
        if t == "presence":
            tg = presence_target(q)
            p = (f"{c} Is any {tg} visible in this image? Reply with exactly one word: present or absent." if tg else
                 f"{c} Question: {q} Reply with exactly one word: present or absent.")
            r = A.ask(fv, p, intent_for(ds, q, t, style="forced")); ans = parse_yesno(r["out"])
        elif t == "count":
            tg = count_target(q); p = f"{c} What is the number of {tg} in this image? {FMT['num']}"
            r = A.ask(fv, p, intent_for(ds, q, t)); ans = normalize_answer(r["out"], ds, t)
        elif t == "comp":
            parts = comp_parts(q)
            p = f"{c} {_swap_comp(parts)} {FMT['yesno']}" if parts else f"{c} Question: {q} Reply with exactly one word: true or false."
            it = dict(k="comp", parts=dict(op={"more": "less", "less": "more"}.get(parts["op"], "equal"), a=parts["b"], b=parts["a"])) if parts else None
            r = A.ask(fv, p, it); ans = parse_yesno(r["out"])
        else:
            p = f"{c} Does this area look more like a city or the countryside? Reply with exactly one word: city or countryside."
            r = A.ask(fv, p, dict(k="ru", style="city")); ans = parse_rural_urban(r["out"])
        return dict(ans=ans, raw=r["out"], prompt=p, calls=[_rec(r)])
    if t.startswith("count"):
        p = f"{c} What is the total number of {fn_count_target(q)} in this image? Reply with a number only."
        r = A.ask(fv, p, intent_for(ds, q, t)); ans = normalize_answer(r["out"], ds, t)
    elif t == "condition":
        sub = cond_subtype(q); a, r, p = _amount_probe(A, fv, sub); ans = cond_to_label(a, sub)
    else:
        parts = fn_yesno_parts(q)
        if parts:
            a, r, p = _amount_probe(A, fv, "road" if parts["obj"] == "road" else "image")
            if a is None: ans = None
            elif parts["quant"] == "entire": ans = _yes(a == parts["polarity"])
            else: ans = _yes(a in (parts["polarity"], "mixed"))
        else:
            p = f"{c} Question: {q} Reply with exactly one word: true or false."
            r = A.ask(fv, p, intent_for(ds, q, t, style="truefalse")); ans = parse_yesno(r["out"])
    return dict(ans=ans, raw=r["out"], prompt=p, calls=[_rec(r)])

# ---------- grounding (shared detector + instance-level verification) ----------
def _merge(boxes, iou=0.5):
    keep = []
    for b in sorted(boxes, key=lambda b: -(b[2] - b[0]) * (b[3] - b[1])):
        if all(_iou(b, k) < iou for k in keep): keep.append(b)
    return keep

def _iou(a, b):
    ix, iy = max(0, min(a[2], b[2]) - max(a[0], b[0])), max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy; u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u > 0 else 0

def src_ground(A, G, ds, q, t, fv, tv, gate):
    views = list(tv.values()) if gate else [fv]
    if ds == "rsvqa_lr":
        if t == "rural_urban": return None
        calls, boxes, ans = [], {}, None
        if t == "presence":
            tg = presence_target(q); subj = relational_subject(tg); ph = subj or tg
            if ph:
                r = G.detect(fv, ph); calls.append(_rec(r)); boxes[ph] = r["out"]; n = len(r["out"])
                ans = ("no" if n == 0 else None) if subj else _yes(n > 0)
        elif t == "count":
            tg = count_target(q)
            if tg: r = G.detect(fv, tg); calls.append(_rec(r)); boxes[tg] = r["out"]; ans = count_bin(len(r["out"]))
        else:
            p = comp_parts(q)
            if p:
                ra, rb = G.detect(fv, p["a"]), G.detect(fv, p["b"]); calls += [_rec(ra), _rec(rb)]
                boxes[p["a"]], boxes[p["b"]] = ra["out"], rb["out"]; ans = _cmp(len(ra["out"]), len(rb["out"]), p["op"])
        return dict(ans=ans, boxes=boxes, calls=calls)
    if not t.startswith("count"): return None
    obj = fn_object(q); calls, allb = [], []
    for v in views:
        r = G.detect(v, obj); calls.append(_rec(r)); allb += r["out"]
    allb = _merge(allb); n = len(allb); attr = fn_attr(q)
    if t == "count_simple" or attr is None:
        return dict(ans=str(n), boxes={obj: allb}, labels=None, calls=calls)
    chosen = sorted(allb, key=lambda b: -(b[2] - b[0]) * (b[3] - b[1]))[:MAX_BOXES]; labels = []
    for k, b in enumerate(chosen):                                   # instance-level verification
        bv = box_view(fv, b, k)
        r = A.ask(bv, f"This close-up is from a UAV (drone) image. Is this {obj} standing in or surrounded "
                      f"by flood water? Reply with exactly one word: flooded or dry.", dict(k="box_flooded"))
        calls.append(_rec(r)); labels.append(parse_instance(r["out"]))
    if chosen and any(labels):
        frac = np.mean([l == attr for l in labels if l]); est = int(round(frac * n))
    else:
        est = 0 if n == 0 else None
    return dict(ans=None if est is None else str(est), boxes={obj: chosen}, labels=labels, n_det=n, calls=calls)

def answer_question(A, G, row, fv, tv_native, tv_up, gate, with_blind=False):
    ds, q, t = row["ds"], row["question"], row["route"]
    r1 = A.ask(fv, f"{q}\n{E1_SUFFIX}", intent_for(ds, q, t))
    tv = tv_native if ds == "floodnet" else tv_up                    # E3 baseline tiles (always computed)
    rec = dict(qid=row["qid"], gate=bool(gate),
               e1=dict(ans=normalize_answer(r1["out"], ds, t, q), raw=r1["out"], calls=[_rec(r1)]),
               sources=dict(full=src_full(A, ds, q, t, fv), tiles=src_tiles(A, ds, q, t, tv),
                            probe=src_probe(A, ds, q, t, fv), ground=src_ground(A, G, ds, q, t, fv, tv_native, gate)))
    if with_blind:
        rb = A.ask(BLANK_VIEW, prompt_structured(ds, q, t), intent_for(ds, q, t))
        rec["blind"] = dict(ans=normalize_answer(rb["out"], ds, t, q), raw=rb["out"], calls=[_rec(rb)])
    return rec

## Run inference
For every question the run stores the answers of **all evidence sources** with their call costs: E1, full image (= E2), tiles (= E3), contrastive probe, grounding, and the question-only baseline. AeroVerify, AeroVerify-Lite and every E5 ablation are then computed **after** the run from this evidence, so they need no extra GPU time. Re-running resumes from the call cache.

In [ ]:
def run_model(name):
    A, G = load_models(name)
    meta = dict(model=name, static_mem_mb=A.static_memory_mb())
    if HAS_CUDA and not DRY_RUN:                                     # warm-up (not cached)
        r0 = SAMPLE.iloc[0]; w = full_view(r0.ds, r0.img)
        for _ in range(2): A._query(w, "Describe this image in one word.")
    out_path, n = PRED_DIR / f"{name}.jsonl", 0
    with open(out_path, "w") as fh:
        def run_group(ds, img, g, pert, level, blind):
            nonlocal n
            fv = full_view(ds, img, pert, level)
            gate = zoom_gain(fv.native) >= GATE_MIN_ZOOM
            tv_native = tile_views(ds, img, pert, level) if ds == "floodnet" else {}
            tv_up = tile_views(ds, img, pert, level, upscale_to=LR_CROP_SIZE) if ds == "rsvqa_lr" else {}
            for _, row in g.iterrows():
                rec = answer_question(A, G, row, fv, tv_native, tv_up, gate, with_blind=blind)
                rec.update(model=name, ds=ds, img=str(img), role=row["role"], pert=pert, level=level)
                fh.write(json.dumps(rec) + "\n"); n += 1
        for ds in DATASETS:
            S = SAMPLE[SAMPLE.ds == ds]
            for img, g in tqdm(list(S.groupby("img", sort=False)), desc=f"{name} · {DS_SHORT[ds]} clean"):
                run_group(ds, img, g, "clean", 0.0, True)
        if RUN_ROBUSTNESS:
            for ds, imgs in ROBUST_IMGS.items():
                S = SAMPLE[(SAMPLE.ds == ds) & SAMPLE.img.isin(imgs)]
                for kind, levels in PERTURBATIONS.items():
                    for level in levels:
                        for img, g in tqdm(list(S.groupby("img", sort=False)), desc=f"{name} · {kind} {level}", leave=False):
                            run_group(ds, img, g, kind, level, False)
    meta.update(records=n, new_calls=A.cache.misses, cache_hits=A.cache.hits)
    (PRED_DIR / f"{name}_meta.json").write_text(json.dumps(meta))
    print(f"{name}: {n} records · {A.cache.misses} new calls · {A.cache.hits} cache hits → {out_path}")
    free_models(A, G)

if "run" in STAGES:
    for m in MODELS_TO_RUN:
        t0 = time.time(); run_model(m); print(f"   {m} took {(time.time() - t0) / 60:.1f} min")
else:
    print("run stage not selected")

## Analysis
**AeroVerify decision rule.** Each evidence source *s* (full image, native tiles, probe, grounding) gets a reliability weight from its DEV accuracy *a<sub>s</sub>* for that question type: *w<sub>s</sub>* = log[(K−1)·a<sub>s</sub>/(1−a<sub>s</sub>)], clipped at 0, so a source no better than chance is ignored. This is the optimal weighted vote for independent voters (Nitzan & Paroush, 1982).
* **Categorical answers:** the posterior is p(ℓ) ∝ exp Σ<sub>s</sub> w<sub>s</sub>·1[v<sub>s</sub>=ℓ]. The answer is the arg-max and the **confidence** is its posterior probability.
* **Counts (FloodNet):** the answer is the weighted median of the source counts. The confidence is the weight share within ±max(1, 10 %) of that median.
* **Tiles** take part only when the **resolution gate** is open (tile side ≥ model input). AeroVerify **abstains** below τ, with τ tuned on DEV.
* **AeroVerify-Lite** asks the full image and the probe first. If they agree it stops; otherwise it gathers the remaining evidence.

All variants and ablations are recomputed here from the stored evidence.

In [ ]:
%%stage analyze
# ── Load evidence, learn reliability weights on DEV, fuse ─────────────────────
MODELS = [m for m in MODELS_TO_ANALYZE if (PRED_DIR / f"{m}.jsonl").exists()]
assert MODELS, f"No predictions in {PRED_DIR} — run the 'run' stage first."
META = {m: json.loads((PRED_DIR / f"{m}_meta.json").read_text()) for m in MODELS}
MODEL_LABEL = {m: MODEL_SPECS[m]["label"] for m in MODELS}
QI = SAMPLE.set_index("qid")
SRC = ["full", "tiles", "probe", "ground"]

def label_set(ds, t, q):
    if t == "condition":
        return list(COND_LABELS[cond_subtype(q)])
    return {"yesno": ["yes", "no"], "count_bin": BIN_KEYS, "cat": ["rural", "urban"]}.get(kind_of(ds, t))

def tol(y): return max(1, round(0.1 * y))
def _ok(x): return x is not None and not (isinstance(x, float) and np.isnan(x))

recs = []
for m in MODELS:
    with open(PRED_DIR / f"{m}.jsonl") as f:
        for line in f:
            r = json.loads(line); qi = QI.loc[r["qid"]]
            d = dict(model=m, ds=r["ds"], role=r["role"], pert=r["pert"], level=r["level"], qid=r["qid"],
                     img=r["img"], img_key=f"{r['ds']}:{r['img']}", type=qi.type, route=qi.route, kind=qi.kind,
                     label=qi.label, subgroup=qi.subgroup, small=bool(qi.small), relational=bool(qi.relational),
                     question=qi.question, gate=r["gate"])
            for s in ["e1", "blind"] + SRC:
                v = r.get(s) if s in ("e1", "blind") else r["sources"].get(s)
                d[f"{s}_ans"] = v["ans"] if v else None
                d[f"{s}_n"] = len(v["calls"]) if v else 0
                d[f"{s}_lat"] = sum(c["lat"] for c in v["calls"]) if v else 0.0
                d[f"{s}_mem"] = max([c["mem"] for c in v["calls"]], default=np.nan) if v else np.nan
            recs.append(d)
EVD = pd.DataFrame(recs)
EVD["valid_labels"] = [label_set(ds, t, q) for ds, t, q in zip(EVD.ds, EVD.route, EVD.question)]

def _hit(ans, y, kind):
    if ans is None or (isinstance(ans, float) and np.isnan(ans)): return np.nan
    if kind == "count":
        try: return float(abs(int(ans) - int(y)) <= tol(int(y)))
        except (TypeError, ValueError): return np.nan
    return float(ans == y)

# reliability weights per (model, dataset, type, source) from DEV, clean images
W = {}
dev = EVD[(EVD.role == "dev") & (EVD.pert == "clean")]
for (m, ds, t), g in dev.groupby(["model", "ds", "route"]):
    k = kind_of(ds, t); K = 2 if k == "yesno" else (len(BIN_KEYS) if k == "count_bin" else (3 if t == "condition" else 2))
    for s in SRC:
        h = np.array([_hit(a, y, k) for a, y in zip(g[f"{s}_ans"], g.label)], float); h = h[~np.isnan(h)]
        a = (h.sum() + 1) / (len(h) + 2)
        if len(h) == 0: W[(m, ds, t, s)] = 0.0
        else: W[(m, ds, t, s)] = float(np.clip(np.log((K - 1) * a / (1 - a)), 0, 4)) if k != "count" else float(a)
WEIGHTS = pd.Series(W).rename_axis(["model", "ds", "type", "source"]).unstack("source")[SRC]
print("Reliability weights learned on DEV (0 = ignored):"); display(WEIGHTS.round(2))

PRIORITY = {"count": ["ground", "tiles", "full", "probe"], "count_bin": ["tiles", "ground", "full", "probe"],
            "yesno": ["probe", "full", "tiles", "ground"], "cat": ["tiles", "full", "probe", "ground"]}

def fuse_row(r, use, weighted=True, gate=True):
    k = r.kind; prio = [s for s in PRIORITY[k] if s in use]
    votes = {}
    for s in prio:
        if s == "tiles" and gate and not r.gate: continue
        a = getattr(r, f"{s}_ans")
        if not _ok(a): continue
        if k != "count" and a not in r.valid_labels: continue
        votes[s] = a
    if not votes: return None, 0.0
    w = {s: (W.get((r.model, r.ds, r.route, s), 0.0) if weighted else 1.0) for s in votes}
    if k == "count":
        vals = sorted((int(votes[s]), max(w[s], 1e-3)) for s in votes)
        tot, acc = sum(x[1] for x in vals), 0.0
        for v_, w_ in vals:
            acc += w_
            if acc >= tot / 2: med = v_; break
        conf = sum(w[s] for s in votes if abs(int(votes[s]) - med) <= tol(med)) / max(sum(w.values()), 1e-9)
        return str(med), float(conf)
    if not weighted:                                                  # run-1 plurality rule
        c = Counter(votes.values()); top = max(c.values()); cands = {a for a, n in c.items() if n == top}
        a = next(votes[s] for s in prio if s in votes and votes[s] in cands)
        return a, top / len(votes)
    labels = r.valid_labels; sc = {l: sum(w[s] for s in votes if votes[s] == l) for l in labels}
    best = max(sc.values()); cands = [l for l in labels if sc[l] == best]
    a = cands[0] if len(cands) == 1 else next((votes[s] for s in prio if s in votes and votes[s] in cands), cands[0])
    z = np.array([sc[l] for l in labels]); p = np.exp(z - z.max()); p /= p.sum()
    return a, float(p[labels.index(a)])

VARIANTS = {   # name: (sources, reliability weights, resolution gate)
    "E4":          (SRC, True, True),
    "E4 −tiles":   (["full", "probe", "ground"], True, True),
    "E4 −ground":  (["full", "tiles", "probe"], True, True),
    "E4 −probe":   (["full", "tiles", "ground"], True, True),
    "E4 −weights": (SRC, False, True),
    "E4 −gate":    (SRC, True, False),
}

def cost(r, srcs, gate=True):
    ss = [s for s in srcs if not (s == "tiles" and gate and not r.gate)]
    mems = [getattr(r, f"{s}_mem") for s in ss]; mems = [x for x in mems if x == x]
    return (sum(getattr(r, f"{s}_n") for s in ss), sum(getattr(r, f"{s}_lat") for s in ss), max(mems) if mems else np.nan)

rows = []
for r in EVD.itertuples(index=False):
    base = dict(model=r.model, ds=r.ds, role=r.role, pert=r.pert, level=r.level, qid=r.qid, img_key=r.img_key,
                type=r.type, kind=r.kind, label=r.label, subgroup=r.subgroup)
    def add(method, pred, conf, n, lat, mem):
        rows.append(dict(base, method=method, pred=pred, conf=conf, n_calls=n, latency=lat, mem=mem))
    if r.pert == "clean":
        add("majority", MAJORITY.get((r.ds, r.type)), 1.0, 0, 0.0, np.nan)
        add("blind", r.blind_ans, 1.0, r.blind_n, r.blind_lat, r.blind_mem)
    add("E1", r.e1_ans, 1.0, r.e1_n, r.e1_lat, r.e1_mem)
    add("E2", r.full_ans, 1.0, r.full_n, r.full_lat, r.full_mem)
    add("E3", r.tiles_ans, 1.0, r.tiles_n, r.tiles_lat, r.tiles_mem)
    for v, (srcs, wtd, gate) in VARIANTS.items():
        p, c = fuse_row(r, srcs, wtd, gate); add(v, p, c, *cost(r, srcs, gate))
    p, c = fuse_row(r, ["full", "probe"], weighted=False); add("SC", p, c, *cost(r, ["full", "probe"]))   # self-consistency baseline
    # AeroVerify-Lite: stop after full + probe when they agree
    f, pr = r.full_ans, r.probe_ans
    agree = _ok(f) and _ok(pr) and (
        abs(int(f) - int(pr)) <= tol(int(f)) if r.kind == "count" else f == pr)
    if agree:
        p, c = fuse_row(r, ["full", "probe"]); add("Lite", p, c, *cost(r, ["full", "probe"]))
    else:
        p, c = fuse_row(r, SRC); add("Lite", p, c, *cost(r, SRC))
R = pd.DataFrame(rows)
R["valid"] = R.pred.notna()
R["correct"] = (R.pred == R.label) & R.valid

def _to_int(x):
    try: return int(x)
    except (TypeError, ValueError): return np.nan
R["abs_err"] = np.where(R.kind == "count", (R.pred.map(_to_int) - R.label.map(_to_int)).abs(), np.nan)

# ── Type-conditional confidence calibration (fit on DEV, clean images) ──────────
# Raw fused confidences are not comparable across question types (a 0.9 on a count question is far less
# reliable than a 0.9 on a condition question). We fit one Platt scaler per (model, dataset, question type,
# method) on DEV and apply it everywhere. Within a type it is monotone (ranking unchanged); across types it
# turns confidence into an estimate of P(correct). The SAME procedure is applied to every selective baseline.
SELECTIVE = ["E4", "Lite", "SC"] + [v for v in VARIANTS if v != "E4"]

def _logit(p):
    p = np.clip(np.asarray(p, float), 1e-4, 1 - 1e-4); return np.log(p / (1 - p))

def platt_fit(x, y, l2=1e-3):
    """Platt scaling with Platt's smoothed targets and a weak pull towards the identity map (robust on small DEV sets)."""
    from scipy.optimize import minimize
    x = np.asarray(x, float); n1 = y.sum(); n0 = len(y) - n1
    t = np.where(y > 0, (n1 + 1) / (n1 + 2), 1 / (n0 + 2))
    def nll(w):
        z = w[0] * x + w[1]
        return np.mean(np.logaddexp(0, z) - t * z) + l2 * ((w[0] - 1) ** 2 + w[1] ** 2)
    return minimize(nll, np.array([1.0, 0.0]), method="L-BFGS-B", bounds=[(0.0, None), (None, None)]).x   # slope ≥ 0: never reverses ranking

def calibrate(R, methods, min_n=8):
    R = R.copy(); R["conf_raw"] = R.conf.astype(float); CAL = {}
    fit = R[(R.role == "dev") & (R.pert == "clean") & R.method.isin(methods) & R.conf_raw.notna()]
    for (m, ds, meth), g in fit.groupby(["model", "ds", "method"]):
        w_all = platt_fit(_logit(g.conf_raw), g.correct.values.astype(float))
        for t in R[(R.model == m) & (R.ds == ds)].type.unique():
            h = g[g.type == t]
            CAL[(m, ds, meth, t)] = platt_fit(_logit(h.conf_raw), h.correct.values.astype(float)) if len(h) >= min_n else w_all
    for (m, ds, meth, t), w in CAL.items():
        sel = (R.model == m) & (R.ds == ds) & (R.method == meth) & (R.type == t) & R.conf_raw.notna()
        R.loc[sel, "conf"] = 1 / (1 + np.exp(-np.clip(w[0] * _logit(R.loc[sel, "conf_raw"]) + w[1], -30, 30)))
    return R, CAL

R, CAL = calibrate(R, SELECTIVE)

# Baseline: E2 answers ranked only by the DEV accuracy of their question type ("type prior").
# Any confidence signal worth publishing must beat this trivial one.
_pri = (R[(R.role == "dev") & (R.pert == "clean") & (R.method == "E2")]
        .groupby(["model", "ds", "type"]).correct.mean())
TP = R[R.method == "E2"].copy(); TP["method"] = "Type prior"
TP["conf"] = [_pri.get((m, d, t), 0.5) for m, d, t in zip(TP.model, TP.ds, TP.type)]; TP["conf_raw"] = TP.conf
R = pd.concat([R, TP], ignore_index=True)
SELECTIVE = SELECTIVE + ["Type prior"]

# abstention threshold τ per (model, dataset, variant) — DEV only
TAU = {}
for (m, ds, v), g in R[(R.role == "dev") & (R.pert == "clean") & R.method.isin(SELECTIVE)].groupby(["model", "ds", "method"]):
    best = (0.0, -1.0)
    for tau in np.unique(np.round(g.conf.values, 6)):
        a = g.conf >= tau
        if a.mean() < MIN_COVERAGE or a.sum() == 0: continue
        acc = g.correct[a].mean()
        if acc > best[1] + 1e-9: best = (float(tau), float(acc))
    TAU[(m, ds, v)] = best[0]
R["answered"] = True
for (m, ds, v), tau in TAU.items():
    sel = (R.model == m) & (R.ds == ds) & (R.method == v); R.loc[sel, "answered"] = R.loc[sel, "conf"] >= tau - 1e-9
EV = R[(R.role == "eval") & (R.pert == "clean")].copy()
R.to_csv(TAB_DIR / "all_predictions_long.csv", index=False)
print("τ (DEV):", {f"{m}/{DS_SHORT[ds]}/{v}": round(t, 3) for (m, ds, v), t in TAU.items() if v in ("E4", "Lite")})
print(f"{len(R):,} method-question rows; EVAL clean rows: {len(EV):,}")

In [ ]:
%%stage analyze
# ── Metrics, bootstrap CIs, significance, LaTeX tables ────────────────────────
from scipy.stats import binomtest

def macro_f1(y, p, labels):
    f = []
    for c in labels:
        tp = np.sum((p == c) & (y == c)); fp = np.sum((p == c) & (y != c)); fn = np.sum((p != c) & (y == c))
        if tp + fp + fn: f.append(2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f)) if f else np.nan

def summarize(g, selective=False):
    ds = g.ds.iloc[0]; h = g[g.answered] if selective else g; out = {}
    for t in TYPES[ds]:
        s = h[h.type == t]; out[t] = s.correct.mean() * 100 if len(s) else np.nan
    out["OA"] = h.correct.mean() * 100 if len(h) else np.nan
    out["AA"] = np.nanmean([out[t] for t in TYPES[ds]])
    cat = [t for t in TYPES[ds] if TYPES[ds][t] != "count"]
    out["F1"] = np.nanmean([macro_f1(h[h.type == t].label.values, h[h.type == t].pred.fillna("∅").values,
                                     sorted(h[h.type == t].label.unique())) * 100 for t in cat if (h.type == t).any()])
    out["MAE"] = h[h.kind == "count"].abs_err.mean() if (h.kind == "count").any() else np.nan
    out["coverage"] = g.answered.mean() * 100 if selective else 100.0
    yn = h[(h.kind == "yesno") & (h.label == "no")]
    out["FPR"] = (yn.pred == "yes").mean() * 100 if len(yn) else np.nan          # hallucinated "yes"
    yy = h[h.kind == "yesno"]
    out["yes_rate"] = (yy.pred == "yes").mean() * 100 if len(yy) else np.nan
    out["true_yes_rate"] = (yy.label == "yes").mean() * 100 if len(yy) else np.nan
    out["invalid"] = (~g.valid).mean() * 100; out["n"] = len(h)
    return out

def boot_ci(g, B=2000, selective=False, key="OA"):
    h = g[g.answered] if selective else g
    imgs = h.img_key.unique(); ix = h.img_key.map({i: k for k, i in enumerate(imgs)}).values
    rng = np.random.default_rng(SEED); Wb = np.zeros((B, len(imgs)))
    np.add.at(Wb, (np.repeat(np.arange(B), len(imgs)), rng.integers(0, len(imgs), (B, len(imgs))).ravel()), 1)
    res = {}
    for name in ["OA"] + list(TYPES[g.ds.iloc[0]]):
        mask = np.ones(len(h), bool) if name == "OA" else (h.type == name).values
        if not mask.any(): res[name] = (np.nan, np.nan); continue
        c = np.bincount(ix[mask], weights=h.correct.values[mask].astype(float), minlength=len(imgs))
        n = np.bincount(ix[mask], minlength=len(imgs)).astype(float)
        a = (Wb @ c) / np.maximum(Wb @ n, 1e-9) * 100; res[name] = tuple(np.percentile(a, [2.5, 97.5]))
    return res

def mcnemar(m, ds, a, b):
    A_ = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == a)].set_index("qid").correct
    B_ = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == b)].set_index("qid").correct.reindex(A_.index)
    w, l = int((~A_ & B_).sum()), int((A_ & ~B_).sum())
    return w, l, (B_.mean() - A_.mean()) * 100, (binomtest(min(w, l), w + l, 0.5).pvalue if w + l else 1.0)

MAIN = ["majority", "blind", "E1", "E2", "E3", "E4 −A", "Lite −A", "E4", "Lite"]
def sel_of(meth): return meth in ("E4", "Lite")
def base_of(meth): return {"E4 −A": "E4", "Lite −A": "Lite"}.get(meth, meth)

S, CI = {}, {}
for m in MODELS:
    for ds in DATASETS:
        for meth in MAIN:
            g = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == base_of(meth))]
            if g.empty: continue
            S[(m, ds, meth)] = summarize(g, selective=sel_of(meth)); CI[(m, ds, meth)] = boot_ci(g, selective=sel_of(meth))

rows = []
for (m, ds, meth), s in S.items():
    r = dict(dataset=DS_SHORT[ds], model=MODEL_LABEL[m], method=meth)
    r.update({TYPE_LABEL[t]: s[t] for t in TYPES[ds]})
    r.update(OA=s["OA"], OA_CI="[{:.1f}, {:.1f}]".format(*CI[(m, ds, meth)]["OA"]), AA=s["AA"], F1=s["F1"], MAE=s["MAE"],
             coverage=s["coverage"], FPR_yes=s["FPR"], unparseable=s["invalid"])
    rows.append(r)
MAINTAB = pd.DataFrame(rows)
for ds in DATASETS:
    t = MAINTAB[MAINTAB.dataset == DS_SHORT[ds]].dropna(axis=1, how="all")
    t.to_csv(TAB_DIR / f"table_main_{ds}.csv", index=False)
    print(f"Main results — {DS_LABEL[ds]} (EVAL, accuracy %; selective rows = answered questions only)"); display(t.round(1))

sig = []
for m in MODELS:
    for ds in DATASETS:
        tests = [(a, b) for a, b in [("E2", "E4 −A"), ("E1", "E4 −A"), ("E3", "E4 −A"), ("majority", "E4 −A"),
                                     ("E2", "Lite −A"), ("E1", "E2")]]
        res = []
        for a, b in tests:
            w, l, d, p = mcnemar(m, ds, base_of(a) if a != "E4 −A" else "E4", "E4" if b == "E4 −A" else ("Lite" if b == "Lite −A" else b))
            res.append([f"{b} vs {a}", w, l, d, p])
        order = np.argsort([x[-1] for x in res]); run_max = 0
        for rank, i in enumerate(order):
            run_max = max(run_max, min(1.0, res[i][-1] * (len(res) - rank))); res[i].append(run_max)
        for x in res:
            sig.append(dict(dataset=DS_SHORT[ds], model=MODEL_LABEL[m], comparison=x[0], wins=x[1], losses=x[2],
                            delta_pp=x[3], p=x[4], p_holm=x[5], significant=x[5] < 0.05))
SIG = pd.DataFrame(sig); SIG.to_csv(TAB_DIR / "table_significance.csv", index=False)
print("Paired exact McNemar tests (Holm within dataset × model)"); display(SIG.round(4))

def rc_expected(conf, y):
    """Risk–coverage curve with ties broken uniformly at random (exact expectation, no ordering artefacts)."""
    conf = np.round(np.asarray(conf, float), 9); y = np.asarray(y, float); n = len(y)
    o = np.argsort(-conf, kind="stable"); c, yy = conf[o], y[o]
    start = np.r_[0, np.flatnonzero(np.diff(c)) + 1]; size = np.diff(np.r_[start, n])
    blk = np.add.reduceat(yy, start); before = np.r_[0.0, np.cumsum(blk)[:-1]]
    b = np.repeat(np.arange(len(start)), size); pos = np.arange(n) - start[b] + 1
    exp_cum = before[b] + blk[b] / size[b] * pos
    k = np.arange(1, n + 1); return k / n, 1 - exp_cum / k

def aurc(g):
    if len(g) == 0: return np.nan
    return rc_expected(g.conf.values, g.correct.values)[1].mean() * 100

abl = []
for m in MODELS:
    for ds in DATASETS:
        ref = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == "E4")]
        for v in list(VARIANTS) + ["Lite"]:
            g = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == v)]
            if g.empty: continue
            sf, ss = summarize(g), summarize(g, selective=True)
            abl.append(dict(dataset=DS_SHORT[ds], model=MODEL_LABEL[m], variant=v, acc_full=sf["OA"],
                            d_acc=sf["OA"] - summarize(ref)["OA"], AURC=aurc(g), d_AURC=aurc(g) - aurc(ref),
                            acc_answered=ss["OA"], coverage=ss["coverage"], calls=g.n_calls.mean(), latency=g.latency.mean()))
ABL = pd.DataFrame(abl); ABL.to_csv(TAB_DIR / "table_ablation.csv", index=False)
print("E5 ablation — accuracy at full coverage and AURC (lower = better)"); display(ABL.round(2))

eff = []
for m in MODELS:
    for ds in DATASETS:
        for meth in ["blind", "E1", "E2", "E3", "Lite", "E4"]:
            g = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == meth)]
            if g.empty: continue
            eff.append(dict(dataset=DS_SHORT[ds], model=MODEL_LABEL[m], method=meth, OA=summarize(g)["OA"],
                            latency_s=g.latency.mean(), latency_p90=g.latency.quantile(0.9), calls=g.n_calls.mean(),
                            peak_mem_gb=g.mem.max() / 1024, weights_gb=META[m]["static_mem_mb"] / 1024))
EFF = pd.DataFrame(eff); EFF.to_csv(TAB_DIR / "table_efficiency.csv", index=False)
print("E6 efficiency (per question; OA at full coverage)"); display(EFF.round(3))

ROB = pd.DataFrame()
if (R.pert != "clean").any():
    rb = R[(R.role == "eval") & R.method.isin(["E1", "E2", "E3", "E4", "Lite"])]
    out = []
    for ds, imgs in ROBUST_IMGS.items():
        keys = {f"{ds}:{i}" for i in imgs}
        for (m, meth, pert, level), g in rb[rb.img_key.isin(keys)].groupby(["model", "method", "pert", "level"]):
            out.append(dict(ds=ds, model=m, method=meth, pert=pert, level=level, OA=summarize(g)["OA"],
                            OA_sel=summarize(g, selective=True)["OA"], coverage=g.answered.mean() * 100))
    ROB = pd.DataFrame(out)
    clean = ROB[ROB.pert == "clean"].set_index(["ds", "model", "method"]).OA
    ROB["drop_pp"] = [r.OA - clean.get((r.ds, r.model, r.method), np.nan) for r in ROB.itertuples()]
    ROB.to_csv(TAB_DIR / "table_robustness.csv", index=False)
    worst = {k: (min(v) if k in ("altitude", "jpeg") else max(v)) for k, v in PERTURBATIONS.items()}
    w = ROB[[worst.get(r.pert) == r.level for r in ROB.itertuples()]]
    print("E7 accuracy drop at the strongest perturbation (pp)")
    display(w.pivot_table(index=["model", "method"], columns="pert", values="drop_pp").round(1))

# what does AeroVerify abstain on?  (accuracy of E2 / AeroVerify on answered vs abstained questions)
abst = []
for m in MODELS:
    for ds in DATASETS:
        e4 = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == "E4")].set_index("qid")
        e2 = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == "E2")].set_index("qid").reindex(e4.index)
        a = e4.answered.astype(bool)
        abst.append(dict(dataset=DS_SHORT[ds], model=MODEL_LABEL[m], answered_pct=a.mean() * 100,
                         E2_on_answered=e2.correct[a].mean() * 100, E4_on_answered=e4.correct[a].mean() * 100,
                         E2_on_abstained=e2.correct[~a].mean() * 100, E4_on_abstained=e4.correct[~a].mean() * 100,
                         n_abstained=int((~a).sum()), AURC_E4=aurc(e4.reset_index()),
                         AURC_SC=aurc(EV[(EV.model == m) & (EV.ds == ds) & (EV.method == "SC")]),
                         AURC_plurality=aurc(EV[(EV.model == m) & (EV.ds == ds) & (EV.method == "E4 −weights")]),
                         AURC_type_prior=aurc(EV[(EV.model == m) & (EV.ds == ds) & (EV.method == "Type prior")])))
ABST = pd.DataFrame(abst); ABST.to_csv(TAB_DIR / "table_abstention.csv", index=False)

# Selective prediction: AeroVerify vs every confidence baseline (all calibrated on DEV the same way).
# ΔAURC = baseline − AeroVerify (positive = AeroVerify better), 95% CI from a paired image-level bootstrap.
SEL_METHODS = [("E4", "AeroVerify"), ("SC", "Self-consistency (E2+probe)"),
               ("E4 −weights", "Plurality vote (all sources)"), ("Type prior", "E2 + question-type prior")]
def _paired_aurc_ci(a, b, B=1000):
    a = a.sort_values("qid"); b = b.set_index("qid").loc[a.qid].reset_index()
    imgs = a.img_key.values; uniq, inv = np.unique(imgs, return_inverse=True)
    rows_of = [np.flatnonzero(inv == k) for k in range(len(uniq))]; rng = np.random.default_rng(SEED); out = []
    ca, ya, cb, yb = a.conf.values, a.correct.values, b.conf.values, b.correct.values
    for _ in range(B):
        ix = np.concatenate([rows_of[k] for k in rng.integers(0, len(uniq), len(uniq))])
        out.append(rc_expected(cb[ix], yb[ix])[1].mean() - rc_expected(ca[ix], ya[ix])[1].mean())
    return np.percentile(np.array(out) * 100, [2.5, 97.5])
sel_rows = []
for m in MODELS:
    for ds in DATASETS:
        ours = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == "E4")]
        if ours.empty: continue
        for meth, lab in SEL_METHODS:
            g = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == meth)]
            if g.empty: continue
            ss = summarize(g, selective=True); d = aurc(g) - aurc(ours)
            lo, hi = (np.nan, np.nan) if meth == "E4" else _paired_aurc_ci(ours, g, B=BOOT_B if "BOOT_B" in globals() else 1000)
            sel_rows.append(dict(dataset=DS_SHORT[ds], model=MODEL_LABEL[m], method=lab, AURC=aurc(g),
                                 dAURC_vs_ours=np.nan if meth == "E4" else d, ci_lo=lo, ci_hi=hi,
                                 acc_answered=ss["OA"], coverage=ss["coverage"], acc_full=summarize(g)["OA"]))
SELTAB = pd.DataFrame(sel_rows); SELTAB.to_csv(TAB_DIR / "table_selective.csv", index=False)
print("Selective prediction (EVAL). ΔAURC = baseline − AeroVerify, 95% paired bootstrap CI; >0 means AeroVerify better")
display(SELTAB.round(1))
L = [r"\begin{tabular}{lllrrrr}", r"\toprule",
     r"Data & Model & Confidence signal & AURC$\downarrow$ & $\Delta$AURC [95\% CI] & Acc.\ answered & Coverage \\", r"\midrule"]
for r in SELTAB.itertuples():
    dd = "--" if np.isnan(r.dAURC_vs_ours) else f"{r.dAURC_vs_ours:+.1f} [{r.ci_lo:+.1f}, {r.ci_hi:+.1f}]"
    nm = r"\textbf{" + r.method + "}" if r.method == "AeroVerify" else r.method
    L.append(f"{r.dataset} & {r.model} & {nm} & {r.AURC:.1f} & {dd} & {r.acc_answered:.1f} & {r.coverage:.1f}" + r" \\")
L += [r"\bottomrule", r"\end{tabular}"]; (TAB_DIR / "table_selective.tex").write_text("\n".join(L))
print("Abstention analysis (EVAL): accuracy on answered vs abstained questions; AURC of three confidence signals")
display(ABST.round(1))

if "floodnet" in DATASETS:
    cc = EV[(EV.ds == "floodnet") & (EV.kind == "count") & EV.method.isin(["E1", "E2", "E3", "E4"])].copy()
    cc["err"] = (cc.pred.map(_to_int) - cc.label.map(_to_int)).abs()
    CNT = (cc.groupby(["model", "type", "method"]).err
           .agg(exact=lambda e: (e == 0).mean() * 100, within1=lambda e: (e <= 1).mean() * 100, MAE="mean").reset_index())
    CNT.to_csv(TAB_DIR / "table_counting.csv", index=False)
    print("FloodNet counting: exact %, within ±1 %, MAE"); display(CNT.round(2))
    L = [r"\begin{tabular}{lllrrr}", r"\toprule", r"Model & Question & Method & Exact (\%) & $\pm$1 (\%) & MAE$\downarrow$ \\", r"\midrule"]
    for r in CNT.itertuples():
        L.append(f"{MODEL_LABEL[r.model]} & {TYPE_LABEL[r.type]} & {METHOD_LABELS[r.method]} & {r.exact:.1f} & {r.within1:.1f} & {r.MAE:.2f}" + r" \\")
    L += [r"\bottomrule", r"\end{tabular}"]; (TAB_DIR / "table_counting.tex").write_text("\n".join(L))

# per-source diagnostics (EVAL, clean): how often each evidence source answers, and how often it is right
src_rows = []
for (m, ds, t), g in EVD[(EVD.role == "eval") & (EVD.pert == "clean")].groupby(["model", "ds", "route"]):
    for s_ in ["e1"] + SRC:
        a = g[f"{s_}_ans"]; ok = a.map(_ok)
        if not ok.any() and s_ in ("tiles", "ground"): continue
        src_rows.append(dict(dataset=DS_SHORT[ds], model=MODEL_LABEL[m], type=t, source=s_, answered=ok.mean() * 100,
                             accuracy=(a == g.label).mean() * 100, weight=W.get((m, ds, t, s_), np.nan)))
SRCTAB = pd.DataFrame(src_rows); SRCTAB.to_csv(TAB_DIR / "table_sources.csv", index=False)
print("Evidence sources (EVAL): answered %, accuracy %, DEV reliability weight")
display(SRCTAB.pivot_table(index=["dataset", "model", "type"], columns="source", values="accuracy").round(1))

# ---------- LaTeX (booktabs) ----------
def _fmt(v, d=1): return "--" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.{d}f}"
NAME_TEX = {"majority": "Most-common answer", "blind": "Question-only", "E1": "E1 Direct", "E2": "E2 Structured",
            "E3": "E3 Region", "E4 −A": r"\textbf{AeroVerify} (full cov.)", "Lite −A": r"AeroVerify-Lite (full cov.)",
            "E4": r"\textbf{AeroVerify} (selective)", "Lite": "AeroVerify-Lite (selective)"}
def latex_main(ds):
    ts = list(TYPES[ds]); cols = [TYPE_LABEL[t] for t in ts] + ["OA", "AA"] + (["MAE$\\downarrow$"] if ds == "floodnet" else []) + ["Cov."]
    L = [r"\begin{tabular}{ll" + "r" * len(cols) + "}", r"\toprule", "Model & Method & " + " & ".join(cols) + r" \\", r"\midrule"]
    for mi, m in enumerate(MODELS):
        block = [(meth, S[(m, ds, meth)]) for meth in MAIN if (m, ds, meth) in S]
        full_cov = [s for meth, s in block if not sel_of(meth)]
        best = {k: max(s[k] for s in full_cov if not np.isnan(s[k])) for k in ts + ["OA", "AA"]}
        for j, (meth, s) in enumerate(block):
            cells = []
            for k in ts + ["OA", "AA"]:
                txt = _fmt(s[k]); cells.append(r"\textbf{" + txt + "}" if (not sel_of(meth) and abs(s[k] - best[k]) < 1e-9) else txt)
            if ds == "floodnet": cells.append(_fmt(s["MAE"], 2))
            cells.append(_fmt(s["coverage"], 0))
            L.append((MODEL_LABEL[m] if j == 0 else "") + " & " + NAME_TEX[meth] + " & " + " & ".join(cells) + r" \\")
        if mi < len(MODELS) - 1: L.append(r"\midrule")
    L += [r"\bottomrule", r"\end{tabular}"]
    (TAB_DIR / f"table_main_{ds}.tex").write_text("\n".join(L))

def latex_ablation():
    L = [r"\begin{tabular}{lrrrr}", r"\toprule", r"Variant & $\Delta$Acc. (pp) & AURC$\downarrow$ & Calls & Latency (s) \\", r"\midrule"]
    for ds in DATASETS:
        for m in MODELS:
            L.append(r"\multicolumn{5}{l}{\textit{" + DS_SHORT[ds] + " / " + MODEL_LABEL[m] + r"}} \\")
            for r in ABL[(ABL.dataset == DS_SHORT[ds]) & (ABL.model == MODEL_LABEL[m])].itertuples():
                name = {"E4": r"AeroVerify (full)"}.get(r.variant, r.variant.replace("E4 −", r"\quad $-$ ").replace("Lite", "AeroVerify-Lite"))
                L.append(f"{name} & {r.d_acc:+.1f} & {r.AURC:.1f} & {r.calls:.1f} & {r.latency:.2f}" + r" \\")
    L += [r"\bottomrule", r"\end{tabular}"]
    (TAB_DIR / "table_ablation.tex").write_text("\n".join(L))

for ds in DATASETS: latex_main(ds)
latex_ablation()
print("LaTeX tables →", sorted(p.name for p in TAB_DIR.glob("*.tex")))

## Figures (paper-ready)
Each figure is a vector PDF at IEEE double-column width (7.16″) with text ≥ 7 pt and embedded fonts, plus a 600-dpi PNG. Colour is fixed per method: E1 orange, E2 green, E3 amber, **AeroVerify blue**, AeroVerify-Lite violet. Baselines are grey.

### Fig. 1 · Method overview (drawn on a real EVAL image)

In [ ]:
%%stage analyze
# Fig. 1 — AeroVerify-VLM pipeline (drawn on a real example)
import textwrap
DS0 = DATASETS[0]
ex = EVD[(EVD.ds == DS0) & (EVD.role == "eval") & (EVD.pert == "clean") & (EVD.type == ("count_complex" if DS0 == "floodnet" else "count"))]
ex = ex.assign(_lab=ex.label.map(_to_int)).sort_values("_lab", ascending=False)        # a busy scene illustrates best
ex_q = ex.qid.iloc[min(2, len(ex) - 1)] if len(ex) else EV[EV.ds == DS0].qid.iloc[0]
ex_row = QI.loc[ex_q]; ex_img = get_image(DS0, ex_row.img)

fig = plt.figure(figsize=(IEEE_2COL, 2.75)); ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 100); ax.set_ylim(0, 40); ax.axis("off")
def box(x, y, w, h, title, body, fc, ec, tc=INK, bold_title=True):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.25,rounding_size=1.2", fc=fc, ec=ec, lw=0.8))
    ax.text(x + 1.2, y + h - 1.4, title, ha="left", va="top", fontsize=7.6, fontweight="bold" if bold_title else "normal", color=tc)
    if body: ax.text(x + 1.2, y + h - 3.9, body, ha="left", va="top", fontsize=7, color=tc if tc != INK else INK2, linespacing=1.15)
def arrow(x0, y0, x1, y1, c=MUTED):
    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=8, lw=0.9, color=c, shrinkA=0, shrinkB=0))

# input image with tile grid
thumb = ex_img.copy(); thumb.thumbnail((600, 600))
iax = fig.add_axes([0.012, 0.16, 0.19, 0.62]); iax.imshow(thumb); iax.set_xticks([]); iax.set_yticks([]); iax.grid(False)
for s in iax.spines.values(): s.set_color(AXISC)
tw, th = thumb.size
iax.axvline(tw / 2, color="white", lw=0.9); iax.axhline(th / 2, color="white", lw=0.9)
ax.text(1.2, 39.6, "Input", fontsize=8, fontweight="bold", va="top")
ax.text(1.2, 37.0, "UAV image + question", fontsize=7, color=INK2, va="top")
ax.text(1.2, 4.6, textwrap.fill("Q: " + ex_row.question, 34), fontsize=7, color=INK, va="top", style="italic")

lanes = [("Full view (E2)", "whole image, downsampled;\ntype-specific prompt"),
         ("Native tiles (E3)", "2×2 full-resolution tiles;\nused only if zoom gain ≥ 1"),
         ("Contrastive probe", "forced choice / decomposed\n(no yes–no tokens)"),
         ("Grounding", "detect instances, verify each\n(e.g. flooded or dry?)")]
y0s = [28.4, 19.6, 10.8, 2.0]
for (t, b), y in zip(lanes, y0s):
    box(30, y, 25, 7.4, t, b, "#eef4fc", BLUE_RAMP[2]); arrow(22.5, 20, 29.6, y + 3.7)
box(60.5, 11.5, 18.5, 17, "Reliability-\nweighted fusion", None, METHOD_COLORS["E4"], METHOD_COLORS["E4"], tc="white")
ax.text(69.75, 19.9, r"$w_s=\log\,\frac{(K-1)\,a_s}{1-a_s}$", ha="center", va="center", fontsize=8.5, color="white")
ax.text(69.75, 16.6, "$a_s$: DEV accuracy\nof source $s$", ha="center", va="center", fontsize=6.8, color="white")
ax.text(69.75, 13.0, "then calibrated per\nquestion type (DEV)", ha="center", va="center", fontsize=6.8, color="white", style="italic")
for y in y0s: arrow(55.6, y + 3.7, 60.2, 20)
box(84, 22, 15, 9.5, "Answer", "+ confidence\n+ evidence boxes", "#eef4fc", BLUE_RAMP[2])
box(84, 8.5, 15, 9.5, "Abstain", "confidence < τ\n(τ tuned on DEV)", BAND, AXISC)
arrow(79.3, 22, 83.7, 26.5, METHOD_COLORS["E4"]); arrow(79.3, 18, 83.7, 13.5, MUTED)
ax.text(30, 39.6, "Evidence sources", fontsize=8, fontweight="bold", va="top")
ax.text(60.5, 39.6, "Decision", fontsize=8, fontweight="bold", va="top")
ax.text(84, 39.6, "Output", fontsize=8, fontweight="bold", va="top")
ax.text(84, 6.4, "Lite: exit early\nwhen full view +\nprobe agree", fontsize=6.8, color=METHOD_COLORS["Lite"], va="top")
savefig(fig, "fig01_pipeline")

### Fig. 2 · Qualitative evidence
Each column shows one EVAL question (picked automatically: E2 wrong, AeroVerify right): tile answers, detected instances coloured by their verification, the vote of each source with its DEV reliability weight *w*, and the final confidence *p*.

In [ ]:
%%stage analyze
# Fig. 2 — Evidence-grounded answers on real UAV images (primary model)
import textwrap
PRIMARY = "moondream2" if "moondream2" in MODELS else MODELS[0]
DSQ = "floodnet" if "floodnet" in DATASETS else DATASETS[0]
SRC_NAME = {"full": "Full view", "tiles": "Tiles", "probe": "Probe", "ground": "Grounding"}

CLEAN_REC = {}
with open(PRED_DIR / f"{PRIMARY}.jsonl") as f:
    for line in f:
        r = json.loads(line)
        if r["pert"] == "clean" and r["role"] == "eval" and r["ds"] == DSQ: CLEAN_REC[r["qid"]] = r

def _score(q):
    """Prefer examples that are easy to read: several sources right, and for counts a correct grounding source."""
    r, qi = CLEAN_REC.get(q), QI.loc[q]
    if r is None: return -99
    srcs = r["sources"]; ok = lambda s_: srcs.get(s_) is not None and str(srcs[s_]["ans"]) == str(qi.label)
    sc = sum(ok(s_) for s_ in SRC)
    if qi.kind == "count":
        sc += 3 * ok("ground") + (1 if 2 <= _to_int(qi.label) <= 12 else 0)
    return sc

def _pick_examples(m, ds, k=3):
    """One example per question type: prefer E2 wrong → AeroVerify right, then AeroVerify right, then anything;
    within a tier, the most legible example (see _score)."""
    e = EV[(EV.model == m) & (EV.ds == ds)].pivot_table(index="qid", columns="method", values="correct", aggfunc="first")
    e4 = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == "E4")].set_index("qid")
    ans = e4.answered.reindex(e.index).astype(bool).values
    tiers = [e.index[e["E4"].astype(bool) & ~e["E2"].astype(bool) & ans], e.index[e["E4"].astype(bool) & ans], e.index]
    order = ["count_complex", "condition", "yesno", "count_simple"] if ds == "floodnet" else ["presence", "count", "comp", "rural_urban"]
    picks, used_imgs = [], set()
    for t in order:
        for tier in tiers:
            c = [q for q in tier if QI.loc[q, "type"] == t and QI.loc[q, "img"] not in used_imgs]
            if c:
                best = max(c, key=_score); picks.append(best); used_imgs.add(QI.loc[best, "img"]); break
        if len(picks) == k: break
    return picks[:k]

def _fmt_ans(a, kind):
    if a is None or (isinstance(a, float) and np.isnan(a)): return "–"
    if kind == "count_bin": return COUNT_BINS[BIN_KEYS.index(a)]
    return {MIXED: "partial", "mixed": "partial", NONFLOODED: "dry"}.get(a, str(a))

ex = _pick_examples(PRIMARY, DSQ)
fig = plt.figure(figsize=(IEEE_2COL, 3.75))
gs = fig.add_gridspec(2, len(ex), height_ratios=[1, 0.72], hspace=0.04, wspace=0.1)
recmap = {q: CLEAN_REC[q] for q in ex}
USED = set()
for j, qid in enumerate(ex):
    rec, qi = recmap[qid], QI.loc[qid]
    im = get_image(DSQ, qi.img); Wn, Hn = im.size; th = im.copy(); th.thumbnail((900, 900)); sx = th.size[0] / Wn
    ax = fig.add_subplot(gs[0, j]); ax.imshow(th); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s_ in ax.spines.values(): s_.set_visible(False)
    if rec["gate"] or DSQ == "rsvqa_lr":
        for n, (x0, y0, x1, y1) in tile_boxes(im.size).items():
            ax.add_patch(Rectangle((x0 * sx, y0 * sx), (x1 - x0) * sx, (y1 - y0) * sx, fill=False, ec="white", lw=0.8)); USED.add("tiles")
            v = rec["sources"]["tiles"]["per"].get(n)
            if v is not None:
                v = "|".join(map(str, v)) if isinstance(v, list) else {MIXED: "partial", "mixed": "partial", NONFLOODED: "dry"}.get(v, str(v))
                ax.text(x0 * sx + 6, y0 * sx + 6, v, fontsize=7, va="top", color=INK,
                        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
    g = rec["sources"].get("ground")
    if g and g.get("boxes"):
        labs = g.get("labels") or []
        for i, b in enumerate(next(iter(g["boxes"].values()))[:60]):
            lab = labs[i] if i < len(labs) else None
            col = NEG if lab == FLOODED else (POS if lab == NONFLOODED else "#ffd34d")   # yellow = detected, not verified
            USED.add({NEG: "flooded", POS: "dry"}.get(col, "detected"))
            ax.add_patch(Rectangle((b[0] * sx, b[1] * sx), (b[2] - b[0]) * sx, (b[3] - b[1]) * sx, fill=False, ec=col, lw=1.0))
    ax.set_title(textwrap.fill(qi.question, 33), fontsize=7.4, fontweight="normal", loc="center", color=INK, pad=3)
    ax = fig.add_subplot(gs[1, j]); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    e4 = EV[(EV.model == PRIMARY) & (EV.method == "E4") & (EV.qid == qid)].iloc[0]
    lines = [("Truth", "", _fmt_ans(qi.label, qi.kind), INK, True)]
    for s_ in SRC:
        v = rec["sources"].get(s_)
        if v is None or (s_ == "tiles" and not rec["gate"]): continue
        a = v["ans"]; w = W.get((PRIMARY, DSQ, qi.route, s_), 0)
        agree = a is not None and str(a) == str(e4.pred)
        lines.append((SRC_NAME[s_], f"w={w:.1f}", _fmt_ans(a, qi.kind), METHOD_COLORS["E4"] if agree else MUTED, False))
    verdict = "abstain" if not e4.answered else _fmt_ans(e4.pred, qi.kind) + (" ✓" if e4.correct else " ✗")
    lines.append(("AeroVerify", f"p={e4.conf:.2f}", verdict, METHOD_COLORS["E4"], True))
    for i, (k_, w_, v_, c_, b_) in enumerate(lines):
        y = 0.97 - i * 0.16
        ax.text(0.02, y, k_, fontsize=7.2, color=INK2, va="top", fontweight="bold" if b_ else "normal")
        ax.text(0.43, y, w_, fontsize=7, color=MUTED, va="top", ha="left")
        ax.text(0.98, y, v_, fontsize=7.2, color=c_, va="top", ha="right", fontweight="bold" if b_ else "normal")
    ax.plot([0.02, 0.98], [0.97 - (len(lines) - 1) * 0.16 + 0.035] * 2, color=GRIDC, lw=0.6)
_tile = Rectangle((0, 0), 1, 1, fill=False, ec=MUTED, lw=1, label="2×2 native tiles (tile answer)")
handles = [h for key, h in [("tiles", _tile),
                            ("flooded", Line2D([], [], color=NEG, lw=1.2, label="detected · verified flooded")),
                            ("dry", Line2D([], [], color=POS, lw=1.2, label="detected · verified dry")),
                            ("detected", Line2D([], [], color="#ffd34d", lw=1.2, label="detected"))] if key in USED]
handles.append(Line2D([], [], color=METHOD_COLORS["E4"], lw=0, marker="s", ms=5, label="agrees with final answer"))
fig.legend(handles=handles, loc="lower center", ncol=len(handles), bbox_to_anchor=(0.5, -0.01), fontsize=6.8)
savefig(fig, "fig02_qualitative", f"{MODEL_LABEL[PRIMARY]}; w = reliability weight, p = calibrated confidence, partial = partly flooded, dry = non flooded")

### Fig. 3 · Main results (full coverage, 95% image-level bootstrap CIs)

In [ ]:
%%stage analyze
# Fig. 3 — Main results: accuracy per question type with 95% CIs (full coverage)
MAIN_M = [("E1", "E1"), ("E2", "E2"), ("E3", "E3"), ("E4", "E4")]
nd, nm = len(DATASETS), len(MODELS)
heights = [len(TYPES[ds]) + 2 for ds in DATASETS]
fig, axes = plt.subplots(nd, nm, figsize=(IEEE_2COL, 0.42 * sum(heights) + 0.9), squeeze=False,
                         gridspec_kw=dict(height_ratios=heights, hspace=0.3, wspace=0.1, top=0.875), sharex=True)
for i, ds in enumerate(DATASETS):
    cats = ["OA", "AA"] + list(TYPES[ds]); yc = np.arange(len(cats))[::-1]; offs = np.linspace(0.27, -0.27, 4)
    for j, m in enumerate(MODELS):
        ax = axes[i, j]
        for (meth, key), off in zip(MAIN_M, offs):
            g = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == meth)]
            s, ci = summarize(g), boot_ci(g)
            for y, c in zip(yc, cats):
                if c in ci and not np.isnan(ci[c][0]):
                    ax.plot(ci[c], [y + off] * 2, color=METHOD_COLORS[key], lw=1.1, alpha=0.5, solid_capstyle="round")
                ax.plot(s[c], y + off, METHOD_MARKERS[key], color=METHOD_COLORS[key], ms=4.8, mec="white", mew=0.6, zorder=3)
        for meth, mk in [("majority", "|"), ("blind", "x")]:
            s = summarize(EV[(EV.model == m) & (EV.ds == ds) & (EV.method == meth)])
            for y, c in zip(yc, cats):
                ax.plot(s[c], y, mk, color=BASELINE_COLOR, ms=6.5 if mk == "|" else 4.5, mew=1.1, zorder=2)
        for y in yc[::2]: ax.axhspan(y - 0.5, y + 0.5, color=BAND, zorder=0, lw=0)
        ax.axhline(yc[1] - 0.5, color=AXISC, lw=0.6)
        ax.set_yticks(yc, [TYPE_LABEL[c] for c in cats] if j == 0 else [""] * len(cats)); ax.grid(axis="y", visible=False)
        ax.set_ylim(-0.6, len(cats) - 0.4); ax.set_xlim(-3, 102); ax.set_xticks([0, 25, 50, 75, 100]); ax.xaxis.set_major_formatter(PCT)
        if j > 0: ax.tick_params(axis="y", length=0)
        ax.set_title(f"{DS_SHORT[ds]} · {MODEL_LABEL[m]}", fontsize=8)
        if i == nd - 1: ax.set_xlabel("accuracy on EVAL questions")
h = [Line2D([], [], marker=METHOD_MARKERS[k], color=METHOD_COLORS[k], lw=1, ms=4.5,
            label=METHOD_LABELS[k] + (" (no abstention)" if k == "E4" else "")) for _, k in MAIN_M]
h += [Line2D([], [], marker="|", color=BASELINE_COLOR, lw=0, ms=7, mew=1.2, label="Most-common answer"),
      Line2D([], [], marker="x", color=BASELINE_COLOR, lw=0, ms=4.5, mew=1.2, label="Question-only")]
fig.legend(handles=h, loc="upper center", ncol=3, bbox_to_anchor=(0.55, 0.995), fontsize=7.5)
savefig(fig, "fig03_main_results", "lines = 95% image-level bootstrap CI")

### Fig. 4 · Selective prediction (risk–coverage)

In [ ]:
%%stage analyze
# Fig. 4 — Selective prediction: risk–coverage curves
def rc_curve(g):
    cov, risk = rc_expected(g.conf.values, g.correct.values); return cov * 100, risk * 100
CURVES = [("E4", "AeroVerify (reliability-weighted)", METHOD_COLORS["E4"], "-"),
          ("E4 −weights", "Plurality vote, all sources (run-1 rule)", MUTED, "-"),
          ("SC", "Self-consistency (E2 + probe)", METHOD_COLORS["E2"], "-"),
          ("Type prior", "E2 + question-type prior", INK2, (0, (3, 1.5)))]
AU_NAMES = ["ours", "plurality", "SC", "type prior"]
fig, axes = plt.subplots(len(DATASETS), len(MODELS), figsize=(IEEE_2COL, 2.15 * len(DATASETS) + 0.4), squeeze=False,
                         gridspec_kw=dict(hspace=0.62, wspace=0.2, top=0.78))
for i, ds in enumerate(DATASETS):
    for j, m in enumerate(MODELS):
        ax = axes[i, j]; ymax = 0; au = []
        for meth, lab, col, ls in CURVES:
            g = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == meth)]
            if g.empty: continue
            cov, risk = rc_curve(g); keep = cov >= 20; au.append(aurc(g))
            ax.plot(cov[keep], risk[keep], linestyle=ls, color=col, lw=1.6 if meth == "E4" else 1.1, label=lab)
            ymax = max(ymax, risk[keep].max())
            if meth == "E4":
                s = summarize(g, selective=True)
                ax.plot(s["coverage"], 100 - s["OA"], "o", ms=7, mfc="none", mec=col, mew=1.2, zorder=5, clip_on=False)
        for k in ["E1", "E2", "E3"]:
            g = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == k)]; rv = (1 - g.correct.mean()) * 100; ymax = max(ymax, rv)
            ax.plot(100, rv, METHOD_MARKERS[k], color=METHOD_COLORS[k], ms=5, mec="white", mew=0.6, zorder=4, clip_on=False)
        ax.set_xlim(20, 101); ax.set_ylim(0, ymax * 1.08 + 1); ax.xaxis.set_major_formatter(PCT); ax.yaxis.set_major_formatter(PCT)
        ax.set_title(f"{DS_SHORT[ds]} · {MODEL_LABEL[m]}", fontsize=8, pad=13)
        ax.text(0.0, 1.015, "AURC↓  " + "  ·  ".join(f"{n} {a:.1f}" for n, a in zip(AU_NAMES, au)), transform=ax.transAxes,
                va="bottom", ha="left", fontsize=6.6, color=INK2)
        if i == len(DATASETS) - 1: ax.set_xlabel("coverage (share of questions answered)")
        if j == 0: ax.set_ylabel("risk (error on answered)")
pts = [Line2D([], [], color=c, lw=1.6 if m_ == "E4" else 1.1, linestyle=ls_, label=l) for m_, l, c, ls_ in CURVES]
pts += [Line2D([], [], marker="o", mfc="none", mec=METHOD_COLORS["E4"], lw=0, ms=6, label="operating point (τ from DEV)")]
pts += [Line2D([], [], marker=METHOD_MARKERS[k], color=METHOD_COLORS[k], lw=0, ms=5, label=METHOD_LABELS[k] + " (answers all)")
        for k in ["E1", "E2", "E3"]]
fig.legend(handles=pts, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0.855), fontsize=7.2)
savefig(fig, "fig04_risk_coverage", "lower curve / lower AURC = better; all confidences calibrated per question type on DEV")

### Fig. 5 · E5 ablation

In [ ]:
%%stage analyze
# Fig. 5 — E5 ablation: change vs full AeroVerify (accuracy at full coverage, and AURC)
ROWS = [("E4 −tiles", "− native tiles"), ("E4 −ground", "− grounding"), ("E4 −probe", "− contrastive probe"),
        ("E4 −weights", "− reliability weights"), ("E4 −gate", "− resolution gate"), ("Lite", "AeroVerify-Lite")]
COLS = [(ds, m) for ds in DATASETS for m in MODELS]
cmap = LinearSegmentedColormap.from_list("div", [NEG, "#f6c9c6", MID, "#c4daf6", POS])
fig, axes = plt.subplots(1, 2, figsize=(IEEE_2COL, 0.32 * len(ROWS) + 1.25), gridspec_kw=dict(wspace=0.06))
for ax, (metric, title, sign) in zip(axes, [("d_acc", "Δ accuracy (pp)  ·  higher is better", 1),
                                            ("d_AURC", "Δ AURC (pp)  ·  lower is better", -1)]):
    M = np.full((len(ROWS), len(COLS)), np.nan)
    for i, (v, _) in enumerate(ROWS):
        for j, (ds, m) in enumerate(COLS):
            r = ABL[(ABL.dataset == DS_SHORT[ds]) & (ABL.model == MODEL_LABEL[m]) & (ABL.variant == v)]
            gate_irrelevant = v == "E4 −gate" and EVD[(EVD.ds == ds)].gate.all()
            if len(r) and not gate_irrelevant: M[i, j] = r[metric].iloc[0]
    lim = max(2.0, np.nanmax(np.abs(M)) if np.isfinite(M).any() else 2.0)
    ax.imshow(sign * M, cmap=cmap, vmin=-lim, vmax=lim, aspect="auto")
    for i in range(len(ROWS)):
        for j in range(len(COLS)):
            if np.isnan(M[i, j]):
                ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fc=BAND, ec="white", lw=1))
                ax.text(j, i, "n/a", ha="center", va="center", fontsize=7, color=MUTED)
            else:
                strong = abs(M[i, j]) > 0.6 * lim
                ax.text(j, i, f"{M[i, j]:+.1f}".replace("-", "−"), ha="center", va="center", fontsize=7.2,
                        color="white" if strong else INK, fontweight="bold" if strong else "normal")
    ax.set_xticks(range(len(COLS)), [f"{DS_SHORT[ds]}\n{MODEL_LABEL[m]}" for ds, m in COLS], fontsize=7)
    ax.set_yticks(range(len(ROWS)), [lab for _, lab in ROWS] if ax is axes[0] else [""] * len(ROWS))
    ax.tick_params(length=0); ax.grid(False)
    for s_ in ax.spines.values(): s_.set_visible(False)
    for x in np.arange(-0.5, len(COLS), 1): ax.axvline(x, color="white", lw=1.5)
    for y in np.arange(-0.5, len(ROWS), 1): ax.axhline(y, color="white", lw=1.5)
    ax.set_title(title, fontsize=8)
savefig(fig, "fig05_ablation", "red = removing the component hurts (it is useful), blue = removing it helps; n/a = gate always open on UAV imagery")

### Fig. 6 · E6 accuracy–latency trade-off

In [ ]:
%%stage analyze
# Fig. 6 — E6 accuracy vs latency (per question) with the Pareto front
EM = [("E1", "E1"), ("E2", "E2"), ("E3", "E3"), ("Lite", "Lite"), ("E4", "E4")]
fig, axes = plt.subplots(1, len(DATASETS), figsize=(IEEE_2COL, 2.45), squeeze=False, gridspec_kw=dict(wspace=0.22))
for ax, ds in zip(axes[0], DATASETS):
    for m, fill in zip(MODELS, ["full", "none"]):
        pts = []
        for meth, key in EM:
            g = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == meth)]
            if g.empty: continue
            x, y = g.latency.mean(), g.correct.mean() * 100; pts.append((x, y))
            ax.plot(x, y, METHOD_MARKERS[key], color=METHOD_COLORS[key], ms=6.5, mec=METHOD_COLORS[key] if fill == "none" else "white",
                    mfc=METHOD_COLORS[key] if fill == "full" else "white", mew=1.1, zorder=3)
        pts.sort(); front, best = [], -1
        for x, y in pts:
            if y > best: front.append((x, y)); best = y
        ax.plot(*zip(*front), color=MUTED, lw=0.8, ls="-" if fill == "full" else (0, (3, 2)), zorder=1)
    ax.set_xscale("log")
    ax.xaxis.set_major_locator(mticker.LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g} s"))
    ax.xaxis.set_minor_formatter(mticker.NullFormatter()); ax.yaxis.set_major_formatter(PCT)
    lo, hi = ax.get_ylim(); ax.set_ylim(lo - 2, hi + 4)
    ax.set_xlabel("latency per question (log scale)"); ax.set_title(DS_LABEL[ds], fontsize=8)
axes[0][0].set_ylabel("overall accuracy (OA)")
h = [Line2D([], [], marker=METHOD_MARKERS[k], color=METHOD_COLORS[k], lw=0, ms=5.5, label=METHOD_LABELS[k]) for _, k in EM]
h += [Line2D([], [], marker="o", color=INK2, mfc=INK2, lw=0.8, ms=5, label=f"{MODEL_LABEL[MODELS[0]]} (filled)")]
if len(MODELS) > 1:
    h += [Line2D([], [], marker="o", color=INK2, mfc="white", lw=0.8, ls=(0, (3, 2)), ms=5, label=f"{MODEL_LABEL[MODELS[1]]} (hollow)")]
fig.legend(handles=h, loc="lower center", ncol=4, bbox_to_anchor=(0.5, 1.0), fontsize=7.2)
savefig(fig, "fig06_efficiency", "RTX 4090, batch 1; lines = Pareto front per model; calls per question in table_efficiency")

### Fig. 7 · E7 robustness to UAV degradations

In [ ]:
%%stage analyze
# Fig. 7 — E7 robustness to UAV-specific degradations
if ROB.empty:
    print("no robustness runs (AV_ROBUSTNESS=0) — skipped")
else:
    PT = {"altitude": ("Altitude", "down-sample factor"), "motion_blur": ("Motion blur", "blur length (px)"),
          "haze": ("Haze", "haze blend"), "jpeg": ("JPEG compression", "quality")}
    RM = [("E1", "E1"), ("E2", "E2"), ("E3", "E3"), ("E4", "E4")]
    ds = next(iter(ROBUST_IMGS))
    fig, axes = plt.subplots(len(MODELS), len(PERTURBATIONS), figsize=(IEEE_2COL, 1.75 * len(MODELS) + 0.45),
                             squeeze=False, sharey="row", gridspec_kw=dict(hspace=0.55, wspace=0.17, top=0.86))
    for i, m in enumerate(MODELS):
        for j, (pert, levels) in enumerate(PERTURBATIONS.items()):
            ax = axes[i, j]; xs = list(range(len(levels) + 1))
            for meth, key in RM:
                d = ROB[(ROB.ds == ds) & (ROB.model == m) & (ROB.method == meth)]
                ys = [d[d.pert == "clean"].OA.mean()] + [d[(d.pert == pert) & np.isclose(d.level, lv)].OA.mean() for lv in levels]
                ax.plot(xs, ys, "-" + METHOD_MARKERS[key], color=METHOD_COLORS[key], lw=1.3, ms=4, mec="white", mew=0.5)
            ax.set_xticks(xs, ["clean"] + [f"{lv:g}" for lv in levels]); ax.set_xlim(-0.3, len(levels) + 0.3)
            ax.yaxis.set_major_formatter(PCT)
            if i == 0: ax.set_title(PT[pert][0], fontsize=8)
            ax.set_xlabel(PT[pert][1], fontsize=7, labelpad=1)
            if j == 0: ax.set_ylabel(f"{MODEL_LABEL[m]}\nOA")
    fig.legend(handles=[Line2D([], [], marker=METHOD_MARKERS[k], color=METHOD_COLORS[k], ms=4.5,
                               label=METHOD_LABELS[k] + (" (no abstention)" if k == "E4" else "")) for _, k in RM],
               loc="lower center", ncol=4, bbox_to_anchor=(0.5, 0.88), fontsize=7.5)
    savefig(fig, "fig07_robustness", f"{DS_LABEL[ds]}, {len(ROBUST_IMGS[ds])} EVAL images per condition")

### Fig. 8 · Yes-bias and hallucinated "yes"

In [ ]:
%%stage analyze
# Fig. 8 — Answer bias on yes/no questions: predicted "yes" rate vs truth, and hallucinated "yes"
BM = [("blind", None), ("E1", "E1"), ("E2", "E2"), ("E3", "E3"), ("E4", "E4")]
fig, axes = plt.subplots(1, 2, figsize=(IEEE_2COL, 0.95 + 0.3 * len(BM) * len(MODELS) * len(DATASETS) / 2),
                         sharey=True, gridspec_kw=dict(wspace=0.08))
blocks = [(ds, m) for ds in DATASETS for m in MODELS]
for ax, (metric, title) in zip(axes, [("yes_rate", "(a) Share of “yes” answers"), ("FPR", "(b) Hallucinated “yes” (truth: “no”)")]):
    yy, ticks = 0.0, []
    for ds, m in blocks:
        ax.text(0, yy + 0.05, f"{DS_SHORT[ds]} · {MODEL_LABEL[m]}", transform=ax.get_yaxis_transform(), ha="left",
                va="bottom", fontsize=7.4, fontweight="bold")
        yy -= 0.75; top = yy
        for meth, key in BM:
            g = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == meth)]
            v = summarize(g)[metric]; col = METHOD_COLORS[key] if key else BASELINE_COLOR
            ax.barh(yy, v, height=0.7, color=col)
            ax.annotate(f"{v:.0f}%", (v, yy), xytext=(3, 0), textcoords="offset points", va="center", fontsize=7, color=INK2)
            ticks.append((yy, METHOD_LABELS[meth] + (" (no abst.)" if meth == "E4" else ""))); yy -= 1
        if metric == "yes_rate":
            tv = summarize(EV[(EV.model == m) & (EV.ds == ds) & (EV.method == "E2")])["true_yes_rate"]
            ax.plot([tv, tv], [yy + 0.5, top + 0.5], color=INK, lw=0.9, ls=(0, (3, 2)))
        yy -= 0.6
    ax.set_yticks([p for p, _ in ticks], [l for _, l in ticks]); ax.grid(axis="y", visible=False)
    ax.set_xlim(0, 112); ax.xaxis.set_major_formatter(PCT); ax.set_ylim(yy + 0.5, 0.9); ax.set_title(title, fontsize=8)
savefig(fig, "fig08_answer_bias", "dashed line in (a) = true share of “yes”; yes/no-type questions only")

### Fig. 9 · Counting on UAV imagery

In [ ]:
%%stage analyze
# Fig. 9 — Counting on UAV imagery: predicted vs true count
if "floodnet" not in DATASETS:
    print("FloodNet not selected — skipped")
else:
    pairs = [(m, meth) for m in MODELS for meth in ("E2", "E4")]
    fig, axes = plt.subplots(1, len(pairs), figsize=(IEEE_2COL, 2.05), squeeze=False, sharey=True, gridspec_kw=dict(wspace=0.08))
    cnt = EV[(EV.ds == "floodnet") & (EV.kind == "count")]
    vmax = float(cnt.label.astype(float).max()) + 2
    rng = np.random.default_rng(0)
    for ax, (m, meth) in zip(axes[0], pairs):
        g = cnt[(cnt.model == m) & (cnt.method == meth)].copy()
        g["y"] = g.label.astype(float); g["p"] = g.pred.map(_to_int).astype(float)
        for t, col, mk in [("count_simple", BLUE_RAMP[4], "o"), ("count_complex", METHOD_COLORS["E1"], "^")]:
            h = g[(g.type == t) & g.p.notna()]
            ax.scatter(h.y + rng.uniform(-0.25, 0.25, len(h)), h.p.clip(upper=vmax) + rng.uniform(-0.25, 0.25, len(h)),
                       s=9, marker=mk, color=col, alpha=0.65, lw=0, label=TYPE_LABEL[t])
        ax.plot([0, vmax], [0, vmax], color=INK, lw=0.7, ls=(0, (3, 2)))
        ax.set_xlim(-1, vmax); ax.set_ylim(-1, vmax); ax.set_aspect("equal", adjustable="box")
        n_clip = int((g.p > vmax).sum())
        if n_clip: ax.text(0.97, 0.03, f"{n_clip} above frame", transform=ax.transAxes, ha="right", fontsize=6.8, color=MUTED)
        mae = g.abs_err.mean(); exact = g.correct.mean() * 100
        ax.text(0.04, 0.96, f"MAE {mae:.2f}\nexact {exact:.0f}%", transform=ax.transAxes, va="top", fontsize=7.2,
                bbox=dict(fc="white", ec="none", alpha=0.85, pad=1.5))
        ax.set_title(f"{MODEL_LABEL[m]}\n{'E2 Structured' if meth == 'E2' else 'AeroVerify'}", fontsize=7.8, loc="center")
        ax.set_xlabel("true count")
    axes[0][0].set_ylabel("predicted count")
    hh, ll = axes[0][0].get_legend_handles_labels()
    fig.legend(hh, ll, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.16), fontsize=7.2, markerscale=1.4)
    savefig(fig, "fig09_counting", "FloodNet-VQA counting questions; dashed = perfect")

### Fig. 10 · Confidence calibration

In [ ]:
%%stage analyze
# Fig. 10 — Is AeroVerify's confidence calibrated? Reliability diagram + ECE
def reliability(g, bins=6, min_n=5):
    e = np.linspace(0, 1, bins + 1); idx = np.clip(np.digitize(g.conf, e) - 1, 0, bins - 1); out = []
    for b in range(bins):
        h = g[idx == b]
        if len(h): out.append((h.conf.mean(), h.correct.mean(), len(h)))
    ece = sum(n * abs(c - a) for c, a, n in out) / max(len(g), 1)
    out = [o for o in out if o[2] >= min_n]
    return out, ece * 100
fig, axes = plt.subplots(1, len(DATASETS), figsize=(IEEE_2COL, 2.4), squeeze=False, gridspec_kw=dict(wspace=0.18))
shades = [METHOD_COLORS["E4"], BLUE_RAMP[5]]; mks = ["o", "s"]
for ax, ds in zip(axes[0], DATASETS):
    ax.plot([0, 1], [0, 1], color=INK, lw=0.7, ls=(0, (3, 2)), label="perfect calibration")
    for m, c, mk in zip(MODELS, shades, mks):
        g = EV[(EV.model == m) & (EV.ds == ds) & (EV.method == "E4")]
        pts, ece = reliability(g)
        _, ece_raw = reliability(g.assign(conf=g.conf_raw))
        x, y, n = zip(*pts)
        ax.plot(x, y, "-", color=c, lw=1.2, label=f"{MODEL_LABEL[m]} (ECE {ece_raw:.1f} → {ece:.1f})")
        ax.scatter(x, y, s=[8 + 60 * k / max(n) for k in n], marker=mk, color=c, edgecolor="white", lw=0.5, zorder=3)
    ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1f}"))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1f}"))
    ax.set_xlabel("AeroVerify confidence (calibrated on DEV)"); ax.set_title(DS_LABEL[ds], fontsize=8); ax.legend(loc="lower right", fontsize=7)
axes[0][0].set_ylabel("accuracy")
savefig(fig, "fig10_calibration", "ECE (%) before → after type-conditional Platt scaling fit on DEV; marker area ∝ questions per bin")

In [ ]:
%%stage analyze
# Fig. 11 — What does AeroVerify abstain on?  Accuracy of E2 and AeroVerify on answered vs abstained questions
blocks = [(ds, m) for ds in DATASETS for m in MODELS]
fig, ax = plt.subplots(figsize=(IEEE_2COL, 2.3))
x = np.arange(len(blocks)); w = 0.2
bars = [("E2_on_answered", "E2 · questions AeroVerify answers", METHOD_COLORS["E2"], None),
        ("E4_on_answered", "AeroVerify · answered", METHOD_COLORS["E4"], None),
        ("E2_on_abstained", "E2 · questions AeroVerify abstains on", METHOD_COLORS["E2"], "////"),
        ("E4_on_abstained", "AeroVerify · abstained (if forced)", METHOD_COLORS["E4"], "////")]
for k, (col, lab, c, hatch) in enumerate(bars):
    vals = [ABST[(ABST.dataset == DS_SHORT[ds]) & (ABST.model == MODEL_LABEL[m])][col].iloc[0] for ds, m in blocks]
    ax.bar(x + (k - 1.5) * w, vals, width=w * 0.9, color=c if hatch is None else "white", edgecolor=c,
           hatch=hatch, lw=1.0, label=lab)
    for xi, v in zip(x + (k - 1.5) * w, vals):
        if v == v: ax.text(xi, v + 1.2, f"{v:.0f}", ha="center", va="bottom", fontsize=6.8, color=INK2)
for xi, (ds, m) in zip(x, blocks):                                     # τ from DEV may answer everything
    if ABST[(ABST.dataset == DS_SHORT[ds]) & (ABST.model == MODEL_LABEL[m])].n_abstained.iloc[0] == 0:
        ax.text(xi + w, 4, "τ answers\nall questions", ha="center", va="bottom", fontsize=6.6, color=MUTED, style="italic")
ax.set_xticks(x, [f"{DS_SHORT[ds]} · {MODEL_LABEL[m]}\n({ABST[(ABST.dataset == DS_SHORT[ds]) & (ABST.model == MODEL_LABEL[m])].answered_pct.iloc[0]:.0f}% answered)"
                  for ds, m in blocks], fontsize=7.2)
ax.set_ylim(0, 105); ax.yaxis.set_major_formatter(PCT); ax.set_ylabel("accuracy"); ax.grid(axis="x", visible=False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.2), ncol=2, fontsize=7.2)
savefig(fig, "fig11_abstention", "abstained questions are those that even the best single prompt mostly gets wrong")

## Summary

In [ ]:
%%stage analyze
# ── Headline numbers + file index (also written to outputs_v2/summary.txt) ────
import contextlib, io
_buf = io.StringIO()
with contextlib.redirect_stdout(_buf):
    for ds in DATASETS:
        print(f"\n{DS_LABEL[ds]}")
        for m in MODELS:
            s2, s4, s4s, sl = S[(m, ds, "E2")], S[(m, ds, "E4 −A")], S[(m, ds, "E4")], S[(m, ds, "Lite −A")]
            t = SIG[(SIG.dataset == DS_SHORT[ds]) & (SIG.model == MODEL_LABEL[m]) & (SIG.comparison == "E4 −A vs E2")].iloc[0]
            lat = {k: EV[(EV.model == m) & (EV.ds == ds) & (EV.method == k)].latency.mean() for k in ("E2", "E4", "Lite")}
            print(f"  {MODEL_LABEL[m]:10s} OA  E2 {s2['OA']:.1f} → AeroVerify {s4['OA']:.1f} ({t.delta_pp:+.1f} pp, Holm p={t.p_holm:.2g})"
                  f" → {s4s['OA']:.1f} on {s4s['coverage']:.0f}% answered | Lite {sl['OA']:.1f}"
                  f" | hallucinated yes {s2['FPR']:.0f}% → {s4['FPR']:.0f}%"
                  f" | latency {lat['E2']:.2f}s → {lat['E4']:.2f}s (Lite {lat['Lite']:.2f}s)")
    print("\nSelective prediction (all confidences calibrated per question type on DEV). ΔAURC = baseline − ours, 95% CI:")
    print(SELTAB.round(1).to_string(index=False))
    print("\nEvidence-source accuracy on EVAL (%):")
    print(SRCTAB.pivot_table(index=["dataset", "model", "type"], columns="source", values="accuracy").round(1).to_string())
    print("\nAnswered (%):")
    print(SRCTAB.pivot_table(index=["dataset", "model", "type"], columns="source", values="answered").round(0).to_string())
    print("\nCondition label space:", COND_LABELS)
print(_buf.getvalue()); (OUT_DIR / "summary.txt").write_text(_buf.getvalue())
print("\nFigures:", FIG_DIR); [print("  ", p.name) for p in sorted(FIG_DIR.glob("*.pdf"))]
print("Tables :", TAB_DIR); [print("  ", p.name) for p in sorted(TAB_DIR.glob("*")) if p.suffix in (".tex", ".csv")]
if DRY_RUN: print("\n⚠ DRY RUN — synthetic data and mock models; these numbers only test the pipeline.")